15.458 Project D Code Notebook
Christine Sinn

Section 1 Part 1: Preparing the Datasets

In [1]:
#Import necessary packages
import pandas as pd

In [2]:
#Load the raw SQL data to transform it
news_train = pd.read_csv("news.csv")
news_test = pd.read_csv("news_test.csv")
news_topics = pd.read_csv("news_topics.csv")
topics = pd.read_csv("topics.csv")

h1 = pd.read_csv("h1.csv")
h2 = pd.read_csv("h2.csv")
h3 = pd.read_csv("h3.csv")
h4 = pd.read_csv("h4.csv")
h5 = pd.read_csv("h5.csv")

In [3]:
#Cleaning - to be safe, expected to be all clean
news_train["article"] = news_train["article"].fillna("").astype(str)
news_test["article"] = news_test["article"].fillna("").astype(str)
news_topics["cat"] = news_topics["cat"].astype(str)
news_topics = news_topics[["id", "cat"]].drop_duplicates()

# print(news_train.head())
# print(news_test.head())
# print(news_topics.head())
# print(topics.head())
# print(h3.head())

In [4]:
#Build the input X tables first. Each model should input the bag of words for all articles (X) and output the multi-class classification into all topics (Y)
X_train = news_train.copy()
X_test = news_test.copy()

#Use article ids for building Y tables
train_ids = X_train["id"]
test_ids = X_test["id"]

In [5]:
#Build the output Y tables next. Transform news_topic into multi-class labels

#Create sets of topic classification labels in each granular level in the hierarchy using h1-h5
topic_labels = {
    "h1": set(h1["h1"].astype(str)),
    "h2": set(h2["h2"].astype(str)),
    "h3": set(h3["h3"].astype(str)),
    "h4": set(h4["h4"].astype(str)),
    "h5": set(h5["h5"].astype(str))
}

#Function to create a wide "one-hot" embedding matrix for a specific hierarchy of topic labels
def build_y_mat(news_topics_df, valid_topics, ids):
    """
    Input:
        news_topics_df (one-to-many data frame of news articles and topic classification labels)
        valid_topics (set of valid topic labels that will be assessed for classification, e.g. valid if an "h3" label)
        ids (news article ids)
    Output:
        Y (multi-label binary matrix with rows as article ids, columns as 0/1 label classification)
    """
    filtered_df = news_topics_df[news_topics_df["cat"].isin(valid_topics)].copy() #only consider the valid label topics
    filtered_df["value"] = 1 #Mark these labels as 1 in the one-hot vector notation

    y = filtered_df.pivot(index = "id", columns = "cat", values = "value").fillna(0) #Rest of labels the articles are not classified to, mark 0
    y = y.reindex(ids, fill_value = 0) #makes all article ids present, even if the articles don't have any classified labels at this hierarchy, so the y tables are same size for each level
    y = y.astype(int)
    y = y.reindex(sorted(y.columns), axis = 1)

    return y

#Dictionary of labels for the training and testing article ids. Each key is a hierarchy (h1 - h5)
Y_train = {}
Y_test = {}

for level, valid_topics in topic_labels.items():
    Y_train[level] = build_y_mat(news_topics, valid_topics, train_ids)
    Y_test[level] = build_y_mat(news_topics, valid_topics, test_ids)

In [6]:
#Audit unsupported labels across the full taxonomy
support_rows = []
unsupported_by_level = {}

for level in ["h1", "h2", "h3", "h4", "h5"]:
    train_support = Y_train[level].sum(axis = 0)
    test_support = Y_test[level].sum(axis = 0)

    level_support = pd.DataFrame({
        "train_positive_count": train_support.astype(int),
        "test_positive_count": test_support.astype(int)
    })
    level_support["unsupported_in_train"] = level_support["train_positive_count"] == 0
    level_support["unsupported_in_test"] = level_support["test_positive_count"] == 0
    level_support["unsupported_in_both"] = (
        level_support["unsupported_in_train"] & level_support["unsupported_in_test"]
    )

    unsupported_by_level[level] = level_support
    support_rows.append(level_support.reset_index(names = "label").assign(level = level))

support_audit = pd.concat(support_rows, ignore_index = True)
unsupported_labels = support_audit[support_audit["unsupported_in_train"]].copy()
unsupported_both = support_audit[support_audit["unsupported_in_both"]].copy()

print("Unsupported labels audit")
print(f"Labels with zero train positives: {unsupported_labels.shape[0]}")
print(f"Labels with zero train and zero test positives: {unsupported_both.shape[0]}")
print()
print(unsupported_labels[[
    "level",
    "label",
    "train_positive_count",
    "test_positive_count",
    "unsupported_in_test"
]].sort_values(["level", "label"]).to_string(index = False))

Unsupported labels audit
Labels with zero train positives: 2
Labels with zero train and zero test positives: 0

level      label  train_positive_count  test_positive_count  unsupported_in_test
   h2 GMIL                           0                    3                False
   h4 E312                           0                   25                False


In [7]:
#Some printing
print(X_train.shape)
print(X_test.shape)

for level in Y_train:
    print(f'{level} train shape: {Y_train[level].shape}')
    print(f'{level} test shape: {Y_test[level].shape}')
    print(f'{level} number of labels: {Y_train[level].shape[1]}')
    print(f'{level} average number of labels per article: {Y_train[level].sum(axis = 1).mean()}')

#Save tables to CSV as a backup
X_train.to_csv("X_train.csv", index = False)
X_test.to_csv("X_test.csv", index = False)

for level in Y_train:
    Y_train[level].to_csv(f'Y_train_{level}.csv')
    Y_test[level].to_csv(f'Y_test_{level}.csv')

(23149, 2)
(390616, 2)
h1 train shape: (23149, 4)
h1 test shape: (390616, 4)
h1 number of labels: 4
h1 average number of labels per article: 1.170115339755497
h2 train shape: (23149, 22)
h2 test shape: (390616, 22)
h2 number of labels: 22
h2 average number of labels per article: 0.3560844960905439
h3 train shape: (23149, 33)
h3 test shape: (390616, 33)
h3 number of labels: 33
h3 average number of labels per article: 1.021124022635967
h4 train shape: (23149, 43)
h4 test shape: (390616, 43)
h4 number of labels: 43
h4 average number of labels per article: 0.6190332195775196
h5 train shape: (23149, 1)
h5 test shape: (390616, 1)
h5 number of labels: 1
h5 average number of labels per article: 0.017236165709101903


In [8]:
X_train["word_count"] = X_train["article"].apply(lambda x: len(x.split()))
X_test["word_count"] = X_test["article"].apply(lambda x: len(x.split()))

print(X_train["word_count"].describe())
print(X_test["word_count"].describe())

count    23149.000000
mean       120.901896
std        104.868207
min          8.000000
25%         51.000000
50%         91.000000
75%        162.000000
max       3298.000000
Name: word_count, dtype: float64
count    390616.000000
mean        124.180384
std         111.412388
min           4.000000
25%          53.000000
50%          94.000000
75%         167.000000
max        4966.000000
Name: word_count, dtype: float64


Section 1 Part 2: Three Models First Pass Classification (Top-Level Topics)

In [9]:
#General Imports for All Models
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score, multilabel_confusion_matrix

In [10]:
from collections import Counter

all_words = " ".join(X_train["article"]).split()
unique_words = set(all_words)
print(f'There are {len(unique_words)} unique words in the training set.')

There are 47152 unique words in the training set.


In [11]:
#Use TF-IDF to vectorize the bag of words input
vectorizer = TfidfVectorizer(
    max_features = 47152, #max amount of words considered in the classification - using the number of unique words in training
    min_df = 5 #drops rare words that appear less than 5 times
)

X_train_vec = vectorizer.fit_transform(X_train["article"])
X_test_vec = vectorizer.transform(X_test["article"])

#Assign the Y train and test matrices, ensuring they line up with the ids of X train and X test
Y_train_h1 = Y_train["h1"]
Y_train_h1 = Y_train_h1.loc[X_train["id"]]
Y_test_h1 = Y_test["h1"]
Y_test_h1 = Y_test_h1.loc[X_test["id"]]

In [12]:
#MODEL ONE: SVM MODEL

#Imports for SVM Model
from sklearn.svm import LinearSVC

#Train Multi-Label SVM and Find Predictions
svm_model = OneVsRestClassifier(
    LinearSVC()
)

svm_model.fit(X_train_vec, Y_train_h1)
Y_pred = svm_model.predict(X_test_vec)

#Print Diagnostic Results
print(classification_report(Y_test_h1, Y_pred, zero_division=0))
print(accuracy_score(Y_test_h1, Y_pred)) #exact matches, may be low for multi-label classification

mcm_svm = multilabel_confusion_matrix(Y_test_h1, Y_pred)
for i, label in enumerate(Y_test_h1.columns):
    print(f'Confusion matrix for {label}:\n {mcm_svm[i]}')

              precision    recall  f1-score   support

           0       0.93      0.91      0.92    185137
           1       0.85      0.71      0.78     58247
           2       0.94      0.91      0.92    116136
           3       0.93      0.90      0.92     99593

   micro avg       0.92      0.88      0.90    459113
   macro avg       0.91      0.86      0.88    459113
weighted avg       0.92      0.88      0.90    459113
 samples avg       0.93      0.92      0.91    459113

0.8237092182603887
Confusion matrix for CCAT      :
 [[192459  13020]
 [ 15878 169259]]
Confusion matrix for ECAT      :
 [[325248   7121]
 [ 16602  41645]]
Confusion matrix for GCAT      :
 [[267513   6967]
 [ 10900 105236]]
Confusion matrix for MCAT      :
 [[284435   6588]
 [  9679  89914]]


In [13]:
#MODEL TWO: LOGISTIC REGRESSION

#Imports for Logistic Regression / Max Entropy Model
from sklearn.linear_model import LogisticRegression

logit_model = OneVsRestClassifier(
    LogisticRegression(
        l1_ratio = 0, #new version of l2 penalty
        solver = "lbfgs", #recommended as general solver for a broad array of problems
        max_iter = 2000
    )
)

logit_model.fit(X_train_vec, Y_train_h1)
Y_pred_logit = logit_model.predict(X_test_vec)

#Diagnostic Metrics
print(classification_report(Y_test_h1, Y_pred_logit, zero_division=0))
print(accuracy_score(Y_test_h1, Y_pred_logit))

mcm_logit = multilabel_confusion_matrix(Y_test_h1, Y_pred_logit)
for i, label in enumerate(Y_test_h1.columns):
    print(f'Confusion matrix for {label}:\n {mcm_logit[i]}')


              precision    recall  f1-score   support

           0       0.94      0.91      0.92    185137
           1       0.90      0.62      0.73     58247
           2       0.95      0.89      0.92    116136
           3       0.95      0.86      0.90     99593

   micro avg       0.94      0.86      0.90    459113
   macro avg       0.93      0.82      0.87    459113
weighted avg       0.94      0.86      0.89    459113
 samples avg       0.92      0.90      0.90    459113

0.8193468777520634
Confusion matrix for CCAT      :
 [[194303  11176]
 [ 17031 168106]]
Confusion matrix for ECAT      :
 [[328516   3853]
 [ 22363  35884]]
Confusion matrix for GCAT      :
 [[268580   5900]
 [ 12243 103893]]
Confusion matrix for MCAT      :
 [[286215   4808]
 [ 13495  86098]]


In [14]:
#MODEL THREE: NAIVE BAYES

from sklearn.naive_bayes import MultinomialNB

#Vectorizing with term counts because Naive Bayes is based on word count likelihoods
count_vectorizer = CountVectorizer(
    max_features = 47152,
    min_df = 5
)

X_train_count = count_vectorizer.fit_transform(X_train["article"])
X_test_count = count_vectorizer.transform(X_test["article"])

nb_model = OneVsRestClassifier(
    MultinomialNB(alpha = 1.0) #alpha is a smoothing factor
)

nb_model.fit(X_train_count, Y_train_h1)
Y_pred_nb = nb_model.predict(X_test_count)

#Diagnostic Metrics
print(classification_report(Y_test_h1, Y_pred_nb, zero_division=0))
print(accuracy_score(Y_test_h1, Y_pred_nb))

mcm_nb = multilabel_confusion_matrix(Y_test_h1, Y_pred_nb)
for i, label in enumerate(Y_test_h1.columns):
    print(f'Confusion matrix for {label}:\n {mcm_nb[i]}')

              precision    recall  f1-score   support

           0       0.90      0.91      0.90    185137
           1       0.61      0.79      0.69     58247
           2       0.89      0.90      0.90    116136
           3       0.76      0.95      0.84     99593

   micro avg       0.82      0.90      0.86    459113
   macro avg       0.79      0.89      0.83    459113
weighted avg       0.83      0.90      0.86    459113
 samples avg       0.87      0.94      0.88    459113

0.7116298359514204
Confusion matrix for CCAT      :
 [[185871  19608]
 [ 16108 169029]]
Confusion matrix for ECAT      :
 [[303124  29245]
 [ 12263  45984]]
Confusion matrix for GCAT      :
 [[261822  12658]
 [ 11314 104822]]
Confusion matrix for MCAT      :
 [[260772  30251]
 [  5323  94270]]


Section 1 Part 3: Finding the Most Specific Classification

In [97]:
levels = ["h1", "h2", "h3", "h4", "h5"]

for level in levels:
    Y_train_level = Y_train[level].loc[X_train["id"]]
    Y_test_level = Y_test[level].loc[X_test["id"]]
    num_labels = len(topic_labels[level])
    print(f'Level: {level}, Number of labels: {num_labels}')

    print(f'---{level} SVM Model---')
    svm_model = OneVsRestClassifier(LinearSVC())
    svm_model.fit(X_train_vec, Y_train_level)
    Y_pred_svm = svm_model.predict(X_test_vec)
    print(classification_report(Y_test_level, Y_pred_svm, zero_division=0))

    print(f'---{level} Logistic Regression Model---')
    logit_model = OneVsRestClassifier(LogisticRegression(l1_ratio = 0, solver = "lbfgs", max_iter = 2000))
    logit_model.fit(X_train_vec, Y_train_level)
    Y_pred_logit = logit_model.predict(X_test_vec)
    print(classification_report(Y_test_level, Y_pred_logit, zero_division=0))

    print(f'---{level} Naive Bayes Model---')
    nb_model = OneVsRestClassifier(MultinomialNB(alpha = 1.0))
    nb_model.fit(X_train_count, Y_train_level)
    Y_pred_nb = nb_model.predict(X_test_count)
    print(classification_report(Y_test_level, Y_pred_nb, zero_division = 0))


Level: h1, Number of labels: 4
---h1 SVM Model---
              precision    recall  f1-score   support

           0       0.93      0.91      0.92    185137
           1       0.85      0.71      0.78     58247
           2       0.94      0.91      0.92    116136
           3       0.93      0.90      0.92     99593

   micro avg       0.92      0.88      0.90    459113
   macro avg       0.91      0.86      0.88    459113
weighted avg       0.92      0.88      0.90    459113
 samples avg       0.93      0.92      0.91    459113

---h1 Logistic Regression Model---
              precision    recall  f1-score   support

           0       0.94      0.91      0.92    185137
           1       0.90      0.62      0.73     58247
           2       0.95      0.89      0.92    116136
           3       0.95      0.86      0.90     99593

   micro avg       0.94      0.86      0.90    459113
   macro avg       0.93      0.82      0.87    459113
weighted avg       0.94      0.86      0.89   

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 0 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 1 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 2 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 3 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-pack

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         0
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           5       0.00      0.00      0.00         0
           6       0.00      0.00      0.00         0
           7       0.00      0.00      0.00         0
           8       0.00      0.00      0.00         0
           9       0.00      0.00      0.00         0
          10       0.00      0.00      0.00         0
          11       0.00      0.00      0.00         0
          12       0.94      0.41      0.58     15547
          13       0.90      0.13      0.22      4244
          14       0.93      0.28      0.43     18492
          15       0.97      0.25      0.40      4159
          16       0.90      0.05      0.09      1818
          17       0.97    

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 0 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 1 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 2 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 3 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 4 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-pack

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.00      0.00      0.00         0
           2       0.00      0.00      0.00         0
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           5       0.00      0.00      0.00         0
           6       0.00      0.00      0.00         0
           7       0.00      0.00      0.00         0
           8       0.00      0.00      0.00         0
           9       0.00      0.00      0.00         0
          10       0.00      0.00      0.00         0
          11       0.00      0.00      0.00         0
          12       0.93      0.37      0.53     15547
          13       0.94      0.04      0.07      4244
          14       0.92      0.23      0.37     18492
          15       0.98      0.09      0.16      4159
          16       1.00      0.00      0.00      1818
          17       0.92    

...

              precision    recall  f1-score   support

           0       0.95      0.83      0.89     39848
           1       0.86      0.70      0.77     35612
           2       0.85      0.58      0.69      8939
           3       0.89      0.61      0.72      5576
           4       0.81      0.28      0.42      1258
           5       0.96      0.83      0.89      2828
           6       0.83      0.60      0.70     21003
           7       0.71      0.09      0.15      2281
           8       0.85      0.37      0.51      3568
           9       0.68      0.36      0.47      2093
          10       0.65      0.21      0.32      3239
          11       0.33      0.00      0.00       541
          12       0.88      0.12      0.21       556
          13       0.93      0.67      0.78      4989
          14       0.89      0.61      0.73       994
          15       0.81      0.69      0.74      2693
          16       0.89      0.23      0.36       446
          17       1.00    

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 23 is present in all training examples.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.95      0.79      0.86     39848
           1       0.92      0.58      0.71     35612
           2       0.89      0.36      0.51      8939
           3       0.90      0.41      0.57      5576
           4       0.76      0.03      0.05      1258
           5       0.98      0.54      0.70      2828
           6       0.87      0.43      0.58     21003
           7       1.00      0.00      0.00      2281
           8       0.96      0.07      0.13      3568
           9       0.87      0.15      0.25      2093
          10       0.63      0.09      0.15      3239
          11       0.00      0.00      0.00       541
          12       0.00      0.00      0.00       556
          13       0.98      0.38      0.55      4989
          14       1.00      0.03      0.06       994
          15       0.91      0.43      0.58      2693
          16       0.00      0.00      0.00       446
          17       0.00    

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 23 is present in all training examples.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.58      0.95      0.72     39848
           1       0.34      0.91      0.50     35612
           2       0.29      0.82      0.43      8939
           3       0.37      0.77      0.50      5576
           4       0.16      0.31      0.21      1258
           5       0.30      0.83      0.44      2828
           6       0.35      0.91      0.51     21003
           7       0.14      0.32      0.19      2281
           8       0.28      0.63      0.38      3568
           9       0.13      0.63      0.21      2093
          10       0.14      0.65      0.23      3239
          11       0.00      0.01      0.00       541
          12       0.01      0.04      0.02       556
          13       0.47      0.76      0.58      4989
          14       0.23      0.54      0.32       994
          15       0.29      0.78      0.43      2693
          16       0.00      0.00      0.00       446
          17       0.00    

Section 2: Building an Ensemble Model

V1 Model: Flat Classifier Across All Labels
- Flat OneVsRest Logistic Regression Classifier for all labels - easiest to interpret like confidence quality
- SCut threshold tuning per label with 5-fold cross-validation like Lewis paper - then use this to calculate confidence score
- Ensure parent classifications
- Multinomial NB Fallback when logistic confidence margin is close to 0

In [39]:
#Create a flat Y_train dataset across h1-h5 categories for a flat classifier approach
levels = ["h1", "h2", "h3", "h4", "h5"]
Y_train_flat = pd.concat((Y_train[level] for level in levels), axis = 1) #columns are labels from all hierarchy levels combined
Y_train_flat = Y_train_flat.loc[X_train["id"]]
Y_train_flat = Y_train_flat.reindex(sorted(Y_train_flat.columns), axis = 1)
print(Y_train_flat.shape)

Y_test_flat = pd.concat((Y_test[level] for level in levels), axis = 1)
Y_test_flat = Y_test_flat.loc[X_test["id"]]
Y_test_flat = Y_test_flat.reindex(sorted(Y_test_flat.columns), axis = 1)
print(Y_test_flat.shape)

(23149, 103)
(390616, 103)


In [40]:
#Build a flat logistic regression classifier on all labels
logit_flat = OneVsRestClassifier(
    LogisticRegression(
        solver = "lbfgs",
        max_iter = 2000,
        class_weight = "balanced" #addresses label imbalance in hierarchies by weighting labels inversely to their frequency in training data, so model also predicts rare categories
    ),
    n_jobs = -1 #use all available CPU cores for parallel processing, speeds up training
)

logit_flat.fit(X_train_vec, Y_train_flat)

#Give continuous decision scores for all labels, which will then be applied to a custom trained threshold to give binary classification predictions
train_scores_flat = logit_flat.decision_function(X_train_vec)
test_scores_flat = logit_flat.decision_function(X_test_vec) #has shape (n_samples, n_labels)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 49 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 80 is present in all training examples.
  warnings.warn(


In [41]:
print(train_scores_flat)

[[-4.53186951 -6.15476511 -5.52606703 ... -3.88127403 -3.56045375
   3.74491171]
 [ 3.38284772 -3.46705996 -2.75088633 ... -5.14817069 -4.33353298
  -3.08518453]
 [ 0.59286059 -3.81306315 -3.9067686  ... -5.16186832 -5.79515795
  -5.09354502]
 ...
 [-1.03508308 -0.10748111  0.36123592 ... -5.14214038 -4.56917978
  -3.94524959]
 [-2.84006212 -3.75958215  1.45512744 ... -4.92521191 -4.38534387
  -3.16137269]
 [-1.81651631 -0.88462288 -0.66430036 ... -5.31948304 -5.1610398
  -3.68722331]]


In [42]:
#Build a fallback Naive Bayes classifier model on all labels, to be turned on when logistic model's confidence is low
nb_flat = OneVsRestClassifier(
    MultinomialNB(alpha = 1.0),
    n_jobs = -1
)

nb_flat.fit(X_train_count, Y_train_flat)
#Don't need to predict scores yet because NB is a fallback model

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 49 is present in all training examples.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/multiclass.py:90: UserWarning: Label not 80 is present in all training examples.
  warnings.warn(


,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",MultinomialNB()
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [43]:
#Tune a SCut threshold for each label using 5-fold cross-validation to optimize the average f1 score - aim to improve classification performance for label imbalances
label_thresholds = {} #dictionary of label: threshold
threshold_grid = np.linspace(-6, 4, 201) #potential thresholds to evaluate
kf = KFold(n_splits = 5, shuffle = True, random_state = 15458)

for idx, label in enumerate(Y_train_flat.columns): #for each label
    Y_label = Y_train_flat.iloc[:, idx] #just pulling column for this label
    if Y_label.sum() == 0: #no positive examples for label, skip threshold tuning, set to high values so no prediction of positives
        label_thresholds[label] = 0 #no extra information, default 0
        continue

    avgf1 = np.zeros(len(threshold_grid)) #average f1 score for each threshold

    for train_index, val_index in kf.split(X_train_vec): #split into train indices and validation indices
        X_train_fold, X_val_fold = X_train_vec[train_index], X_train_vec[val_index]
        Y_train_fold, Y_val_fold = Y_label.iloc[train_index], Y_label.iloc[val_index]

        logit_fold = OneVsRestClassifier(
            LogisticRegression(
                solver = "lbfgs",
                max_iter = 2000,
                class_weight = "balanced"
            ),
            n_jobs = -1
        )

        logit_fold.fit(X_train_fold, Y_train_fold)
        val_scores = logit_fold.decision_function(X_val_fold)

        for i, threshold in enumerate(threshold_grid):
            Y_pred = (val_scores >= threshold).astype(int)
            avgf1[i] += f1_score(Y_val_fold, Y_pred, zero_division = 0)

    avgf1 /= kf.get_n_splits()
    label_thresholds[label] = threshold_grid[np.argmax(avgf1)] #threshold with the highest average f1 score across the folds

thresholds = np.array([label_thresholds[label] for label in Y_train_flat.columns]) #making numpy array for later vectorized operations


In [60]:
#Use SCut thresholds to make predictions on test set
Y_pred_flat = (test_scores_flat >= thresholds[np.newaxis, :]).astype(int) #expanding threshold array so it matches dimensions of test_scores

#Implement the fallback NB model when confidence margin is small
conf_margin = 0.001 #units of logits / each classifier's own score scale
nb_preds_flat = nb_flat.predict(X_test_count)

uncertain = np.abs(test_scores_flat - thresholds[np.newaxis, :]) <= conf_margin #boolean array with same shape of test scores: uncertain[i, j] = True means article i for label j has an uncertain score
Y_pred_flat[uncertain] = nb_preds_flat[uncertain] #replace uncertain positions with NB predictions using boolean indexing

In [61]:
#Enforce parent classifications are consistent with child classifications - if classifier classifies a child topic, then the article should also have the parent topic

#Build the parent-child map
parent_map = {}
for child, parent in zip(h2["h2"].astype(str), h2["h1"].astype(str)):
    parent_map[child] = parent
for child, parent in zip(h3["h3"].astype(str), h3["h2"].astype(str)):
    parent_map[child] = parent
for child, parent in zip(h4["h4"].astype(str), h4["h3"].astype(str)):
    parent_map[child] = parent
for child, parent in zip(h5["h5"].astype(str), h5["h4"].astype(str)):
    parent_map[child] = parent

#Enforce parent labels
predictions_df = pd.DataFrame(Y_pred_flat, columns = Y_train_flat.columns, index = X_test["id"])
for child, parent in parent_map.items():
    if child in predictions_df.columns and parent in predictions_df.columns:
        predictions_df[parent] = np.where(predictions_df[child] == 1, 1, predictions_df[parent])

In [62]:
#Confidence scores for each prediction (as a percentage)
confidence_scores = 1 / (1 + np.exp(-(test_scores_flat - thresholds[np.newaxis, :])))
confidence_df = pd.DataFrame(confidence_scores, columns = Y_train_flat.columns, index = X_test["id"])

In [63]:
#Evaluate flat ensemble model with parent-enforced predictions (formatted with Codex)
print("=" * 80)
print("FLAT ENSEMBLE MODEL: OneVsRest Logistic Regression with Parent Enforcement")
print("=" * 80)
print(f"\nTotal labels (all hierarchies): {Y_train_flat.shape[1]}")
print(f"Training set size: {Y_train_flat.shape[0]}")
print(f"Test set size: {Y_test_flat.shape[0]}")
print(f"\nLow-confidence margin for NB fallback: {conf_margin}")
print(f"Number of predictions using NB fallback: {uncertain.sum()}")

# Align y_true and y_pred on the same article ids and label columns before scoring
Y_true_eval = Y_test_flat.loc[predictions_df.index, predictions_df.columns]

print("\n" + "=" * 80)
print("Classification Report")
print("=" * 80)
print(classification_report(Y_true_eval, predictions_df, zero_division=0))

print("\n" + "=" * 80)
print("Sample Predictions with Confidence Scores")
print("=" * 80)
sample_ids = X_test["id"].iloc[:5].tolist()
for article_id in sample_ids:
    row_pred = predictions_df.loc[article_id]
    row_conf = confidence_df.loc[article_id]
    predicted_labels = row_pred[row_pred == 1].index.tolist()
    if not predicted_labels:
        print(f"\nArticle {article_id}: No labels predicted")
    else:
        print(f"\nArticle {article_id} predicted labels:")
        for label in sorted(predicted_labels):
            print(f"  - {label}: confidence {row_conf[label]:.3f}")


FLAT ENSEMBLE MODEL: OneVsRest Logistic Regression with Parent Enforcement

Total labels (all hierarchies): 103
Training set size: 23149
Test set size: 390616

Low-confidence margin for NB fallback: 0.001
Number of predictions using NB fallback: 782193

Classification Report
              precision    recall  f1-score   support

           0       0.44      0.42      0.43     11876
           1       0.66      0.46      0.54      5793
           2       0.48      0.53      0.50     18411
           3       0.50      0.64      0.56      3657
           4       0.88      0.89      0.89     73905
           5       0.90      0.86      0.88     39848
           6       0.83      0.47      0.60     11355
           7       0.73      0.78      0.75     35612
           8       0.78      0.15      0.25       928
           9       0.67      0.76      0.71     20420
          10       0.68      0.68      0.68      8939
          11       0.74      0.74      0.74      5576
          12       0.

V2: Soft Top Down Conditioning Model with Low Support Naive Bayes
Classify separate model banks for each label by depth using: 
- TF-IDF article text
- Parent class's probability score
For every label at every depth:
- SCut threshold fine-tuning
- Add ancestors if child definition turns on
- For low support labels, call fallback NB

In [32]:
from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import StratifiedKFold

normalize_label = lambda value: str(value).strip()
levels = ["h1", "h2", "h3", "h4", "h5"]
parent_level = {"h2": "h1", "h3": "h2", "h4": "h3", "h5": "h4"}
all_level_labels = {
    "h1": sorted({normalize_label(value) for value in h1["h1"]}),
    "h2": sorted({normalize_label(value) for value in h2["h2"]}),
    "h3": sorted({normalize_label(value) for value in h3["h3"]}),
    "h4": sorted({normalize_label(value) for value in h4["h4"]}),
    "h5": sorted({normalize_label(value) for value in h5["h5"]})
}
level_parent_lookup = {
    "h2": dict(zip(h2["h2"].map(normalize_label), h2["h1"].map(normalize_label))),
    "h3": dict(zip(h3["h3"].map(normalize_label), h3["h2"].map(normalize_label))),
    "h4": dict(zip(h4["h4"].map(normalize_label), h4["h3"].map(normalize_label))),
    "h5": dict(zip(h5["h5"].map(normalize_label), h5["h4"].map(normalize_label)))
}
normalized_parent_map = {
    normalize_label(child): normalize_label(parent)
    for child, parent in parent_map.items()
}

low_support_cutoff = 25
low_support_margin = 0.2
max_cv_splits = 5
conditioning_results = []


def build_conditioned_matrix(text_matrix, parent_scores = None, parent_probs = None, parent_pass = None):
    if parent_scores is None:
        return text_matrix

    parent_block = csr_matrix(
        np.column_stack([parent_scores, parent_probs, parent_pass.astype(int)])
    )
    return hstack([text_matrix, parent_block], format = "csr")


def choose_cv_splits(y, max_splits = 5):
    positives = int(y.sum())
    negatives = int(len(y) - positives)
    return min(max_splits, positives, negatives)


def tune_scut_threshold(scores, y_true, grid_size = 81):
    if np.unique(scores).size == 1:
        return float(scores[0])

    lower, upper = np.quantile(scores, [0.02, 0.98])
    if lower == upper:
        lower, upper = float(scores.min()), float(scores.max())

    threshold_grid = np.linspace(lower, upper, grid_size)
    best_threshold = float(threshold_grid[0])
    best_f1 = -1.0

    for threshold in threshold_grid:
        y_pred = (scores >= threshold).astype(int)
        candidate_f1 = f1_score(y_true, y_pred, zero_division = 0)
        if candidate_f1 > best_f1:
            best_f1 = candidate_f1
            best_threshold = float(threshold)

    return best_threshold


def fit_conditioned_label(label, y_train_label, X_train_conditioned, X_test_conditioned):
    y_train_array = y_train_label.to_numpy()
    positive_support = int(y_train_array.sum())
    low_support = positive_support < low_support_cutoff
    result = {
        "label": label,
        "positive_support": positive_support,
        "low_support": low_support,
        "nb_fallback_test_count": 0
    }

    if np.unique(y_train_array).size == 1:
        constant_value = int(y_train_array[0])
        constant_score = 8.0 if constant_value == 1 else -8.0
        result.update({
            "model_type": "constant",
            "threshold": 0.0,
            "train_score": np.full(X_train_conditioned.shape[0], constant_score),
            "test_score": np.full(X_test_conditioned.shape[0], constant_score),
            "train_prob": np.full(X_train_conditioned.shape[0], float(constant_value)),
            "test_prob": np.full(X_test_conditioned.shape[0], float(constant_value)),
            "train_pred": np.full(X_train_conditioned.shape[0], constant_value, dtype = int),
            "test_pred": np.full(X_test_conditioned.shape[0], constant_value, dtype = int)
        })
        return result

    cv_splits = choose_cv_splits(y_train_array, max_cv_splits)
    if cv_splits >= 2:
        skf = StratifiedKFold(n_splits = cv_splits, shuffle = True, random_state = 15458)
        oof_scores = np.zeros(X_train_conditioned.shape[0])

        for train_index, val_index in skf.split(np.zeros(X_train_conditioned.shape[0]), y_train_array):
            fold_model = LogisticRegression(
                solver = "lbfgs",
                max_iter = 2000,
                class_weight = "balanced"
            )
            fold_model.fit(X_train_conditioned[train_index], y_train_array[train_index])
            oof_scores[val_index] = fold_model.decision_function(X_train_conditioned[val_index])

        threshold = tune_scut_threshold(oof_scores, y_train_array)
    else:
        threshold = 0.0

    conditioned_model = LogisticRegression(
        solver = "lbfgs",
        max_iter = 2000,
        class_weight = "balanced"
    )
    conditioned_model.fit(X_train_conditioned, y_train_array)

    train_score = conditioned_model.decision_function(X_train_conditioned)
    test_score = conditioned_model.decision_function(X_test_conditioned)
    train_prob = conditioned_model.predict_proba(X_train_conditioned)[:, 1]
    test_prob = conditioned_model.predict_proba(X_test_conditioned)[:, 1]
    train_pred = (train_score >= threshold).astype(int)
    test_pred = (test_score >= threshold).astype(int)

    if low_support:
        nb_model = MultinomialNB(alpha = 1.0)
        nb_model.fit(X_train_count, y_train_array)

        nb_train_prob = nb_model.predict_proba(X_train_count)[:, 1]
        nb_test_prob = nb_model.predict_proba(X_test_count)[:, 1]
        uncertain_train = np.abs(train_score - threshold) <= low_support_margin
        uncertain_test = np.abs(test_score - threshold) <= low_support_margin

        train_pred[uncertain_train] = (nb_train_prob[uncertain_train] >= 0.5).astype(int)
        test_pred[uncertain_test] = (nb_test_prob[uncertain_test] >= 0.5).astype(int)
        train_prob[uncertain_train] = nb_train_prob[uncertain_train]
        test_prob[uncertain_test] = nb_test_prob[uncertain_test]
        result["nb_fallback_test_count"] = int(uncertain_test.sum())

    result.update({
        "model_type": "conditioned_logit",
        "threshold": threshold,
        "train_score": train_score,
        "test_score": test_score,
        "train_prob": train_prob,
        "test_prob": test_prob,
        "train_pred": train_pred,
        "test_pred": test_pred
    })
    return result


conditioned_train_scores = {}
conditioned_test_scores = {}
conditioned_train_probs = {}
conditioned_test_probs = {}
conditioned_train_pass = {}
conditioned_test_pass = {}

for level in levels:
    Y_train_level = Y_train[level].loc[X_train["id"]].copy()
    Y_test_level = Y_test[level].loc[X_test["id"]].copy()
    Y_train_level.columns = [normalize_label(column) for column in Y_train_level.columns]
    Y_test_level.columns = [normalize_label(column) for column in Y_test_level.columns]
    Y_train_level = Y_train_level.loc[:, ~Y_train_level.columns.duplicated()]
    Y_test_level = Y_test_level.loc[:, ~Y_test_level.columns.duplicated()]
    Y_train_level = Y_train_level.reindex(columns = all_level_labels[level], fill_value = 0)
    Y_test_level = Y_test_level.reindex(columns = all_level_labels[level], fill_value = 0)

    level_train_scores = pd.DataFrame(index = X_train["id"], columns = Y_train_level.columns, dtype = float)
    level_test_scores = pd.DataFrame(index = X_test["id"], columns = Y_train_level.columns, dtype = float)
    level_train_probs = pd.DataFrame(index = X_train["id"], columns = Y_train_level.columns, dtype = float)
    level_test_probs = pd.DataFrame(index = X_test["id"], columns = Y_train_level.columns, dtype = float)
    level_train_pass = pd.DataFrame(index = X_train["id"], columns = Y_train_level.columns, dtype = int)
    level_test_pass = pd.DataFrame(index = X_test["id"], columns = Y_train_level.columns, dtype = int)

    for label in Y_train_level.columns:
        if level == "h1":
            X_train_conditioned = X_train_vec
            X_test_conditioned = X_test_vec
            parent_label = None
        else:
            parent_label = level_parent_lookup[level][label]
            parent_train_scores = conditioned_train_scores[parent_level[level]][parent_label].to_numpy()
            parent_test_scores = conditioned_test_scores[parent_level[level]][parent_label].to_numpy()
            parent_train_probs = conditioned_train_probs[parent_level[level]][parent_label].to_numpy()
            parent_test_probs = conditioned_test_probs[parent_level[level]][parent_label].to_numpy()
            parent_train_pass = conditioned_train_pass[parent_level[level]][parent_label].to_numpy()
            parent_test_pass = conditioned_test_pass[parent_level[level]][parent_label].to_numpy()

            X_train_conditioned = build_conditioned_matrix(
                X_train_vec,
                parent_train_scores,
                parent_train_probs,
                parent_train_pass
            )
            X_test_conditioned = build_conditioned_matrix(
                X_test_vec,
                parent_test_scores,
                parent_test_probs,
                parent_test_pass
            )

        fitted_label = fit_conditioned_label(label, Y_train_level[label], X_train_conditioned, X_test_conditioned)

        level_train_scores[label] = fitted_label["train_score"]
        level_test_scores[label] = fitted_label["test_score"]
        level_train_probs[label] = fitted_label["train_prob"]
        level_test_probs[label] = fitted_label["test_prob"]
        level_train_pass[label] = fitted_label["train_pred"]
        level_test_pass[label] = fitted_label["test_pred"]

        conditioning_results.append({
            "level": level,
            "label": label,
            "parent_label": parent_label,
            "positive_support": fitted_label["positive_support"],
            "low_support": fitted_label["low_support"],
            "threshold": fitted_label["threshold"],
            "model_type": fitted_label["model_type"],
            "nb_fallback_test_count": fitted_label["nb_fallback_test_count"]
        })

    conditioned_train_scores[level] = level_train_scores
    conditioned_test_scores[level] = level_test_scores
    conditioned_train_probs[level] = level_train_probs
    conditioned_test_probs[level] = level_test_probs
    conditioned_train_pass[level] = level_train_pass.astype(int)
    conditioned_test_pass[level] = level_test_pass.astype(int)

soft_topdown_predictions = pd.concat(
    [conditioned_test_pass[level] for level in levels],
    axis = 1
)
soft_topdown_predictions = soft_topdown_predictions.reindex(sorted(soft_topdown_predictions.columns), axis = 1)

soft_topdown_confidence = pd.concat(
    [conditioned_test_probs[level] for level in levels],
    axis = 1
)
soft_topdown_confidence = soft_topdown_confidence.loc[:, soft_topdown_predictions.columns]

for child, parent in normalized_parent_map.items():
    if child in soft_topdown_predictions.columns and parent in soft_topdown_predictions.columns:
        soft_topdown_predictions[parent] = np.where(
            soft_topdown_predictions[child] == 1,
            1,
            soft_topdown_predictions[parent]
        )

soft_topdown_true = pd.concat(
    [Y_test[level].loc[X_test["id"]].rename(columns = normalize_label) for level in levels],
    axis = 1
)
soft_topdown_true = soft_topdown_true.loc[:, ~soft_topdown_true.columns.duplicated()]
soft_topdown_true = soft_topdown_true.reindex(columns = soft_topdown_predictions.columns, fill_value = 0)
conditioning_results_df = pd.DataFrame(conditioning_results)

print("=" * 80)
print("SOFT TOP-DOWN CONDITIONING MODEL")
print("=" * 80)
print(f"Total labels trained: {conditioning_results_df.shape[0]}")
print(f"Low-support cutoff: {low_support_cutoff}")
print(f"Labels using low-support NB fallback: {int(conditioning_results_df['low_support'].sum())}")
print(f"Total test predictions touched by NB fallback: {conditioning_results_df['nb_fallback_test_count'].sum()}")

print("\nOverall Metrics")
print(f"Micro precision: {precision_score(soft_topdown_true, soft_topdown_predictions, average = 'micro', zero_division = 0):.4f}")
print(f"Micro recall: {recall_score(soft_topdown_true, soft_topdown_predictions, average = 'micro', zero_division = 0):.4f}")
print(f"Micro F1: {f1_score(soft_topdown_true, soft_topdown_predictions, average = 'micro', zero_division = 0):.4f}")
print(f"Macro precision: {precision_score(soft_topdown_true, soft_topdown_predictions, average = 'macro', zero_division = 0):.4f}")
print(f"Macro recall: {recall_score(soft_topdown_true, soft_topdown_predictions, average = 'macro', zero_division = 0):.4f}")
print(f"Macro F1: {f1_score(soft_topdown_true, soft_topdown_predictions, average = 'macro', zero_division = 0):.4f}")

print("\nPer-level Metrics")
for level in levels:
    level_true = Y_test[level].loc[X_test["id"]].rename(columns = normalize_label)
    level_true = level_true.loc[:, ~level_true.columns.duplicated()]
    level_true = level_true.reindex(columns = all_level_labels[level], fill_value = 0)
    level_pred = conditioned_test_pass[level].loc[level_true.index, level_true.columns]
    print(
        f"{level}: "
        f"precision = {precision_score(level_true, level_pred, average = 'micro', zero_division = 0):.4f}, "
        f"recall = {recall_score(level_true, level_pred, average = 'micro', zero_division = 0):.4f}, "
        f"F1 = {f1_score(level_true, level_pred, average = 'micro', zero_division = 0):.4f}"
    )


SOFT TOP-DOWN CONDITIONING MODEL
Total labels trained: 116
Low-support cutoff: 25
Labels using low-support NB fallback: 25
Total test predictions touched by NB fallback: 24580

Overall Metrics
Micro precision: 0.5173
Micro recall: 0.8141
Micro F1: 0.6326
Macro precision: 0.3821
Macro recall: 0.6809
Macro F1: 0.4254

Per-level Metrics
h1: precision = 0.9143, recall = 0.8925, F1 = 0.9033
h2: precision = 0.5065, recall = 0.7683, F1 = 0.6105
h3: precision = 0.6455, recall = 0.7474, F1 = 0.6927
h4: precision = 0.4664, recall = 0.7862, F1 = 0.5855
h5: precision = 0.9854, recall = 0.9854, F1 = 0.9854


V3: Soft Top-Down Conditioning Model with Low Support SVM
- V2 model with using SVM instead of NB
- New evaluation point for low support is low positive support (< 250) and low confidence margin (< 0.05)
- Conditional hierarchy trained on parent probability scores from cross-validation folds in training, not parent true labels (to avoid overoptimism in training period)

In [36]:
from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score
import numpy as np
import pandas as pd


# ============================================================
# Basic setup
# ============================================================

normalize_label = lambda value: str(value).strip()

levels = ["h1", "h2", "h3", "h4", "h5"]
parent_level = {"h2": "h1", "h3": "h2", "h4": "h3", "h5": "h4"}

all_level_labels = {
    "h1": sorted({normalize_label(value) for value in h1["h1"]}),
    "h2": sorted({normalize_label(value) for value in h2["h2"]}),
    "h3": sorted({normalize_label(value) for value in h3["h3"]}),
    "h4": sorted({normalize_label(value) for value in h4["h4"]}),
    "h5": sorted({normalize_label(value) for value in h5["h5"]})
}

level_parent_lookup = {
    "h2": dict(zip(h2["h2"].map(normalize_label), h2["h1"].map(normalize_label))),
    "h3": dict(zip(h3["h3"].map(normalize_label), h3["h2"].map(normalize_label))),
    "h4": dict(zip(h4["h4"].map(normalize_label), h4["h3"].map(normalize_label))),
    "h5": dict(zip(h5["h5"].map(normalize_label), h5["h4"].map(normalize_label)))
}

normalized_parent_map = {
    normalize_label(child): normalize_label(parent)
    for child, parent in parent_map.items()
}

low_support_cutoff = 250
low_support_margin = 0.05
max_cv_splits = 5
conditioning_results = []


# ============================================================
# Helpers
# ============================================================

def build_conditioned_matrix(text_matrix, parent_scores=None, parent_probs=None):
    """
    Use only parent score for conditioning.
    parent_probs stays in the signature so existing call sites still work.
    """
    if parent_scores is None:
        return text_matrix

    parent_block = csr_matrix(parent_scores.reshape(-1, 1))
    return hstack([text_matrix, parent_block], format="csr")


def choose_cv_splits(y, max_splits=5):
    positives = int(y.sum())
    negatives = int(len(y) - positives)
    return min(max_splits, positives, negatives)


def tune_scut_threshold(scores, y_true, beta=1.0, grid_size=81):
    """
    SCut-style threshold tuning.
    beta=1.0 -> F1
    beta=0.5 -> F0.5 (more precision-oriented)
    """
    if np.unique(scores).size == 1:
        return float(scores[0])

    lower, upper = np.quantile(scores, [0.02, 0.98])
    if lower == upper:
        lower, upper = float(scores.min()), float(scores.max())

    threshold_grid = np.linspace(lower, upper, grid_size)
    best_threshold = float(threshold_grid[0])
    best_score = -1.0

    for threshold in threshold_grid:
        y_pred = (scores >= threshold).astype(int)
        candidate_score = fbeta_score(
            y_true,
            y_pred,
            beta=beta,
            zero_division=0
        )
        if candidate_score > best_score:
            best_score = candidate_score
            best_threshold = float(threshold)

    return best_threshold


def get_threshold_beta(level):
    """
    Use F1 on upper levels, F0.5 on deeper levels.
    Adjust if you want h2 to be more precision-oriented too.
    """
    if level in ["h1", "h2"]:
        return 1.0
    return 0.5


def safe_sigmoid(x):
    x = np.clip(x, -30, 30)
    return 1.0 / (1.0 + np.exp(-x))


def make_logit_model():
    """
    Removed class_weight='balanced'
    """
    return LogisticRegression(
        solver="lbfgs",
        max_iter=2000
    )


def make_svm_model():
    """
    Removed class_weight='balanced'
    """
    return LinearSVC(
        max_iter=5000
    )


def fit_logistic_oof_with_threshold(
    y_train_array,
    X_train_conditioned,
    X_test_conditioned,
    beta=1.0,
    max_cv_splits=5
):
    """
    Returns OOF train outputs for downstream conditioning,
    plus full-model train/test outputs for final prediction.
    """
    result = {}

    if np.unique(y_train_array).size == 1:
        constant_value = int(y_train_array[0])
        constant_score = 8.0 if constant_value == 1 else -8.0

        result["threshold"] = 0.0
        result["oof_train_score"] = np.full(X_train_conditioned.shape[0], constant_score, dtype=float)
        result["oof_train_prob"] = np.full(X_train_conditioned.shape[0], float(constant_value), dtype=float)

        result["full_train_score"] = np.full(X_train_conditioned.shape[0], constant_score, dtype=float)
        result["full_train_prob"] = np.full(X_train_conditioned.shape[0], float(constant_value), dtype=float)

        result["test_score"] = np.full(X_test_conditioned.shape[0], constant_score, dtype=float)
        result["test_prob"] = np.full(X_test_conditioned.shape[0], float(constant_value), dtype=float)

        result["model_type"] = "constant"
        result["model"] = None
        return result

    cv_splits = choose_cv_splits(y_train_array, max_cv_splits)

    if cv_splits >= 2:
        skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=15458)
        oof_train_score = np.zeros(X_train_conditioned.shape[0], dtype=float)
        oof_train_prob = np.zeros(X_train_conditioned.shape[0], dtype=float)

        for train_index, val_index in skf.split(np.zeros(X_train_conditioned.shape[0]), y_train_array):
            fold_model = make_logit_model()
            fold_model.fit(X_train_conditioned[train_index], y_train_array[train_index])

            oof_train_score[val_index] = fold_model.decision_function(X_train_conditioned[val_index])
            oof_train_prob[val_index] = fold_model.predict_proba(X_train_conditioned[val_index])[:, 1]

        threshold = tune_scut_threshold(oof_train_score, y_train_array, beta=beta)
    else:
        fallback_model = make_logit_model()
        fallback_model.fit(X_train_conditioned, y_train_array)
        oof_train_score = fallback_model.decision_function(X_train_conditioned)
        oof_train_prob = fallback_model.predict_proba(X_train_conditioned)[:, 1]
        threshold = tune_scut_threshold(oof_train_score, y_train_array, beta=beta)

    full_model = make_logit_model()
    full_model.fit(X_train_conditioned, y_train_array)

    full_train_score = full_model.decision_function(X_train_conditioned)
    full_train_prob = full_model.predict_proba(X_train_conditioned)[:, 1]

    test_score = full_model.decision_function(X_test_conditioned)
    test_prob = full_model.predict_proba(X_test_conditioned)[:, 1]

    result["threshold"] = float(threshold)
    result["oof_train_score"] = oof_train_score
    result["oof_train_prob"] = oof_train_prob

    result["full_train_score"] = full_train_score
    result["full_train_prob"] = full_train_prob

    result["test_score"] = test_score
    result["test_prob"] = test_prob

    result["model_type"] = "conditioned_logit"
    result["model"] = full_model
    return result


def fit_svm_oof_with_threshold(
    y_train_array,
    X_train_conditioned,
    X_test_conditioned,
    beta=1.0,
    max_cv_splits=5
):
    """
    SVM fallback used only for low-support labels in the gray zone.
    """
    result = {}

    if np.unique(y_train_array).size == 1:
        constant_value = int(y_train_array[0])
        constant_score = 8.0 if constant_value == 1 else -8.0

        result["threshold"] = 0.0
        result["oof_train_score"] = np.full(X_train_conditioned.shape[0], constant_score, dtype=float)
        result["full_train_score"] = np.full(X_train_conditioned.shape[0], constant_score, dtype=float)
        result["test_score"] = np.full(X_test_conditioned.shape[0], constant_score, dtype=float)
        result["model"] = None
        return result

    cv_splits = choose_cv_splits(y_train_array, max_cv_splits)

    if cv_splits >= 2:
        skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=15458)
        oof_train_score = np.zeros(X_train_conditioned.shape[0], dtype=float)

        for train_index, val_index in skf.split(np.zeros(X_train_conditioned.shape[0]), y_train_array):
            fold_model = make_svm_model()
            fold_model.fit(X_train_conditioned[train_index], y_train_array[train_index])
            oof_train_score[val_index] = fold_model.decision_function(X_train_conditioned[val_index])

        threshold = tune_scut_threshold(oof_train_score, y_train_array, beta=beta)
    else:
        fallback_model = make_svm_model()
        fallback_model.fit(X_train_conditioned, y_train_array)
        oof_train_score = fallback_model.decision_function(X_train_conditioned)
        threshold = tune_scut_threshold(oof_train_score, y_train_array, beta=beta)

    full_model = make_svm_model()
    full_model.fit(X_train_conditioned, y_train_array)

    full_train_score = full_model.decision_function(X_train_conditioned)
    test_score = full_model.decision_function(X_test_conditioned)

    result["threshold"] = float(threshold)
    result["oof_train_score"] = oof_train_score
    result["full_train_score"] = full_train_score
    result["test_score"] = test_score
    result["model"] = full_model
    return result


def fit_conditioned_label_oof(level, label, y_train_label, X_train_conditioned, X_test_conditioned):
    y_train_array = y_train_label.to_numpy().astype(int)
    positive_support = int(y_train_array.sum())
    low_support = positive_support < low_support_cutoff
    threshold_beta = get_threshold_beta(level)

    logit_fit = fit_logistic_oof_with_threshold(
        y_train_array=y_train_array,
        X_train_conditioned=X_train_conditioned,
        X_test_conditioned=X_test_conditioned,
        beta=threshold_beta,
        max_cv_splits=max_cv_splits
    )

    threshold = logit_fit["threshold"]

    # Main predictions from logistic
    train_pred = (logit_fit["full_train_score"] >= threshold).astype(int)
    test_pred = (logit_fit["test_score"] >= threshold).astype(int)

    train_prob = logit_fit["full_train_prob"].copy()
    test_prob = logit_fit["test_prob"].copy()

    svm_threshold = None
    svm_fallback_train_count = 0
    svm_fallback_test_count = 0

    if low_support and logit_fit["model_type"] != "constant":
        svm_fit = fit_svm_oof_with_threshold(
            y_train_array=y_train_array,
            X_train_conditioned=X_train_conditioned,
            X_test_conditioned=X_test_conditioned,
            beta=threshold_beta,
            max_cv_splits=max_cv_splits
        )

        svm_threshold = svm_fit["threshold"]

        uncertain_train = np.abs(logit_fit["full_train_score"] - threshold) <= low_support_margin
        uncertain_test = np.abs(logit_fit["test_score"] - threshold) <= low_support_margin

        svm_train_pred = (svm_fit["full_train_score"] >= svm_threshold).astype(int)
        svm_test_pred = (svm_fit["test_score"] >= svm_threshold).astype(int)

        # Precision-oriented agreement rule inside gray zone
        train_pred[uncertain_train] = (
            (train_pred[uncertain_train] == 1) &
            (svm_train_pred[uncertain_train] == 1)
        ).astype(int)

        test_pred[uncertain_test] = (
            (test_pred[uncertain_test] == 1) &
            (svm_test_pred[uncertain_test] == 1)
        ).astype(int)

        # bookkeeping confidence proxy in gray zone
        train_prob[uncertain_train] = safe_sigmoid(svm_fit["full_train_score"][uncertain_train])
        test_prob[uncertain_test] = safe_sigmoid(svm_fit["test_score"][uncertain_test])

        svm_fallback_train_count = int(uncertain_train.sum())
        svm_fallback_test_count = int(uncertain_test.sum())

    result = {
        "label": label,
        "positive_support": positive_support,
        "low_support": low_support,
        "model_type": logit_fit["model_type"],
        "threshold": float(threshold),
        "svm_threshold": svm_threshold,
        "svm_fallback_train_count": svm_fallback_train_count,
        "svm_fallback_test_count": svm_fallback_test_count,

        # OOF train outputs for downstream stacking
        "oof_train_score": logit_fit["oof_train_score"],
        "oof_train_prob": logit_fit["oof_train_prob"],

        # full-fit train/test outputs for analysis and final prediction
        "train_score": logit_fit["full_train_score"],
        "test_score": logit_fit["test_score"],
        "train_prob": train_prob,
        "test_prob": test_prob,
        "train_pred": train_pred.astype(int),
        "test_pred": test_pred.astype(int)
    }

    return result


# ============================================================
# Full OOF-stacked training over hierarchy
# ============================================================

# These hold OOF train outputs and full test outputs by level.
# Downstream levels MUST use OOF train outputs on the training side.
conditioned_oof_train_scores = {}
conditioned_oof_train_probs = {}

conditioned_train_scores = {}
conditioned_test_scores = {}

conditioned_train_probs = {}
conditioned_test_probs = {}

conditioned_train_pass = {}
conditioned_test_pass = {}

for level in levels:
    Y_train_level = Y_train[level].loc[X_train["id"]].copy()
    Y_test_level = Y_test[level].loc[X_test["id"]].copy()

    Y_train_level.columns = [normalize_label(column) for column in Y_train_level.columns]
    Y_test_level.columns = [normalize_label(column) for column in Y_test_level.columns]

    Y_train_level = Y_train_level.loc[:, ~Y_train_level.columns.duplicated()]
    Y_test_level = Y_test_level.loc[:, ~Y_test_level.columns.duplicated()]

    Y_train_level = Y_train_level.reindex(columns=all_level_labels[level], fill_value=0)
    Y_test_level = Y_test_level.reindex(columns=all_level_labels[level], fill_value=0)

    # OOF training outputs for downstream conditioning
    level_oof_train_scores = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=float)
    level_oof_train_probs = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=float)

    # Full-fit training and test outputs for diagnostics and final predictions
    level_train_scores = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=float)
    level_test_scores = pd.DataFrame(index=X_test["id"], columns=Y_train_level.columns, dtype=float)

    level_train_probs = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=float)
    level_test_probs = pd.DataFrame(index=X_test["id"], columns=Y_train_level.columns, dtype=float)

    level_train_pass = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=int)
    level_test_pass = pd.DataFrame(index=X_test["id"], columns=Y_train_level.columns, dtype=int)

    for label in Y_train_level.columns:
        if level == "h1":
            X_train_conditioned = X_train_vec
            X_test_conditioned = X_test_vec
            parent_label = None
        else:
            parent_label = level_parent_lookup[level][label]

            # TRAIN SIDE: use OOF parent outputs only
            parent_train_scores = conditioned_oof_train_scores[parent_level[level]][parent_label].to_numpy()
            parent_train_probs = conditioned_oof_train_probs[parent_level[level]][parent_label].to_numpy()

            # TEST SIDE: use full-fit parent test outputs
            parent_test_scores = conditioned_test_scores[parent_level[level]][parent_label].to_numpy()
            parent_test_probs = conditioned_test_probs[parent_level[level]][parent_label].to_numpy()

            X_train_conditioned = build_conditioned_matrix(
                X_train_vec,
                parent_scores=parent_train_scores,
                parent_probs=parent_train_probs
            )

            X_test_conditioned = build_conditioned_matrix(
                X_test_vec,
                parent_scores=parent_test_scores,
                parent_probs=parent_test_probs
            )

        fitted_label = fit_conditioned_label_oof(
            level=level,
            label=label,
            y_train_label=Y_train_level[label],
            X_train_conditioned=X_train_conditioned,
            X_test_conditioned=X_test_conditioned
        )

        level_oof_train_scores[label] = fitted_label["oof_train_score"]
        level_oof_train_probs[label] = fitted_label["oof_train_prob"]

        level_train_scores[label] = fitted_label["train_score"]
        level_test_scores[label] = fitted_label["test_score"]

        level_train_probs[label] = fitted_label["train_prob"]
        level_test_probs[label] = fitted_label["test_prob"]

        level_train_pass[label] = fitted_label["train_pred"]
        level_test_pass[label] = fitted_label["test_pred"]

        conditioning_results.append({
            "level": level,
            "label": label,
            "parent_label": parent_label,
            "positive_support": fitted_label["positive_support"],
            "low_support": fitted_label["low_support"],
            "threshold": fitted_label["threshold"],
            "svm_threshold": fitted_label["svm_threshold"],
            "model_type": fitted_label["model_type"],
            "svm_fallback_train_count": fitted_label["svm_fallback_train_count"],
            "svm_fallback_test_count": fitted_label["svm_fallback_test_count"]
        })

    conditioned_oof_train_scores[level] = level_oof_train_scores
    conditioned_oof_train_probs[level] = level_oof_train_probs

    conditioned_train_scores[level] = level_train_scores
    conditioned_test_scores[level] = level_test_scores

    conditioned_train_probs[level] = level_train_probs
    conditioned_test_probs[level] = level_test_probs

    conditioned_train_pass[level] = level_train_pass.astype(int)
    conditioned_test_pass[level] = level_test_pass.astype(int)


# ============================================================
# Final assembled prediction tables
# ============================================================

soft_topdown_predictions = pd.concat(
    [conditioned_test_pass[level] for level in levels],
    axis=1
)
soft_topdown_predictions = soft_topdown_predictions.loc[:, ~soft_topdown_predictions.columns.duplicated()]
soft_topdown_predictions = soft_topdown_predictions.reindex(sorted(soft_topdown_predictions.columns), axis=1)

soft_topdown_confidence = pd.concat(
    [conditioned_test_probs[level] for level in levels],
    axis=1
)
soft_topdown_confidence = soft_topdown_confidence.loc[:, ~soft_topdown_confidence.columns.duplicated()]
soft_topdown_confidence = soft_topdown_confidence.reindex(columns=soft_topdown_predictions.columns)

# Optional hierarchy closure at the very end
for child, parent in normalized_parent_map.items():
    if child in soft_topdown_predictions.columns and parent in soft_topdown_predictions.columns:
        soft_topdown_predictions[parent] = np.where(
            soft_topdown_predictions[child] == 1,
            1,
            soft_topdown_predictions[parent]
        )

soft_topdown_true = pd.concat(
    [Y_test[level].loc[X_test["id"]].rename(columns=normalize_label) for level in levels],
    axis=1
)
soft_topdown_true = soft_topdown_true.loc[:, ~soft_topdown_true.columns.duplicated()]
soft_topdown_true = soft_topdown_true.reindex(columns=soft_topdown_predictions.columns, fill_value=0)

conditioning_results_df = pd.DataFrame(conditioning_results)


# ============================================================
# Reporting
# ============================================================

print("=" * 80)
print("FULL OOF SOFT TOP-DOWN CONDITIONING MODEL")
print("=" * 80)
print(f"Total labels trained: {conditioning_results_df.shape[0]}")
print(f"Low-support cutoff: {low_support_cutoff}")
print(f"Labels marked low-support: {int(conditioning_results_df['low_support'].sum())}")
print(f"Total train predictions touched by SVM fallback: {conditioning_results_df['svm_fallback_train_count'].sum()}")
print(f"Total test predictions touched by SVM fallback: {conditioning_results_df['svm_fallback_test_count'].sum()}")

print("\nOverall Metrics")
print(f"Micro precision: {precision_score(soft_topdown_true, soft_topdown_predictions, average='micro', zero_division=0):.4f}")
print(f"Micro recall: {recall_score(soft_topdown_true, soft_topdown_predictions, average='micro', zero_division=0):.4f}")
print(f"Micro F1: {f1_score(soft_topdown_true, soft_topdown_predictions, average='micro', zero_division=0):.4f}")
print(f"Macro precision: {precision_score(soft_topdown_true, soft_topdown_predictions, average='macro', zero_division=0):.4f}")
print(f"Macro recall: {recall_score(soft_topdown_true, soft_topdown_predictions, average='macro', zero_division=0):.4f}")
print(f"Macro F1: {f1_score(soft_topdown_true, soft_topdown_predictions, average='macro', zero_division=0):.4f}")

print("\nPer-level Metrics")
for level in levels:
    level_true = Y_test[level].loc[X_test["id"]].rename(columns=normalize_label)
    level_true = level_true.loc[:, ~level_true.columns.duplicated()]
    level_true = level_true.reindex(columns=all_level_labels[level], fill_value=0)

    level_pred = conditioned_test_pass[level].loc[level_true.index, level_true.columns]

    print(
        f"{level}: "
        f"precision = {precision_score(level_true, level_pred, average='micro', zero_division=0):.4f}, "
        f"recall = {recall_score(level_true, level_pred, average='micro', zero_division=0):.4f}, "
        f"F1 = {f1_score(level_true, level_pred, average='micro', zero_division=0):.4f}"
    )

FULL OOF SOFT TOP-DOWN CONDITIONING MODEL
Total labels trained: 116
Low-support cutoff: 250
Labels marked low-support: 67
Total train predictions touched by SVM fallback: 3266
Total test predictions touched by SVM fallback: 62366

Overall Metrics
Micro precision: 0.5206
Micro recall: 0.7796
Micro F1: 0.6243
Macro precision: 0.3741
Macro recall: 0.6368
Macro F1: 0.4020

Per-level Metrics
h1: precision = 0.9100, recall = 0.8988, F1 = 0.9044
h2: precision = 0.4529, recall = 0.7573, F1 = 0.5668
h3: precision = 0.6859, recall = 0.6551, F1 = 0.6701
h4: precision = 0.4339, recall = 0.7276, F1 = 0.5436
h5: precision = 0.9738, recall = 0.9738, F1 = 0.9738


V4: Soft Top-Down Conditioning Model with KNN Fallback
- Same as V3 but using KNN as fallback instead of SVM

In [37]:
from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score
from sklearn.preprocessing import normalize
import numpy as np
import pandas as pd


# ============================================================
# Optional but recommended for KNN on text features
# ============================================================
# If you have not already normalized your vectors, do this once:
# X_train_vec = normalize(X_train_vec, norm="l2")
# X_test_vec = normalize(X_test_vec, norm="l2")


# ============================================================
# Basic setup
# ============================================================

normalize_label = lambda value: str(value).strip()

levels = ["h1", "h2", "h3", "h4", "h5"]
parent_level = {"h2": "h1", "h3": "h2", "h4": "h3", "h5": "h4"}

all_level_labels = {
    "h1": sorted({normalize_label(value) for value in h1["h1"]}),
    "h2": sorted({normalize_label(value) for value in h2["h2"]}),
    "h3": sorted({normalize_label(value) for value in h3["h3"]}),
    "h4": sorted({normalize_label(value) for value in h4["h4"]}),
    "h5": sorted({normalize_label(value) for value in h5["h5"]})
}

level_parent_lookup = {
    "h2": dict(zip(h2["h2"].map(normalize_label), h2["h1"].map(normalize_label))),
    "h3": dict(zip(h3["h3"].map(normalize_label), h3["h2"].map(normalize_label))),
    "h4": dict(zip(h4["h4"].map(normalize_label), h4["h3"].map(normalize_label))),
    "h5": dict(zip(h5["h5"].map(normalize_label), h5["h4"].map(normalize_label)))
}

normalized_parent_map = {
    normalize_label(child): normalize_label(parent)
    for child, parent in parent_map.items()
}

low_support_cutoff = 250
low_support_margin = 0.05
max_cv_splits = 5
knn_neighbors = 5

conditioning_results = []


# ============================================================
# Helpers
# ============================================================

def build_conditioned_matrix(text_matrix, parent_scores=None, parent_probs=None):
    """
    Use only parent score for conditioning.
    parent_probs is left in the signature so call sites do not break.
    """
    if parent_scores is None:
        return text_matrix

    parent_block = csr_matrix(parent_scores.reshape(-1, 1))
    return hstack([text_matrix, parent_block], format="csr")


def choose_cv_splits(y, max_splits=5):
    positives = int(y.sum())
    negatives = int(len(y) - positives)
    return min(max_splits, positives, negatives)


def tune_scut_threshold(scores, y_true, beta=1.0, grid_size=81):
    """
    SCut-style threshold tuning.
    beta=1.0 -> F1
    beta=0.5 -> F0.5
    """
    if np.unique(scores).size == 1:
        return float(scores[0])

    lower, upper = np.quantile(scores, [0.02, 0.98])
    if lower == upper:
        lower, upper = float(scores.min()), float(scores.max())

    threshold_grid = np.linspace(lower, upper, grid_size)
    best_threshold = float(threshold_grid[0])
    best_score = -1.0

    for threshold in threshold_grid:
        y_pred = (scores >= threshold).astype(int)
        candidate_score = fbeta_score(
            y_true,
            y_pred,
            beta=beta,
            zero_division=0
        )
        if candidate_score > best_score:
            best_score = candidate_score
            best_threshold = float(threshold)

    return best_threshold


def get_threshold_beta(level):
    """
    Use F1 for upper levels, F0.5 for deeper levels.
    """
    if level in ["h1", "h2"]:
        return 1.0
    return 0.5


def make_logit_model():
    return LogisticRegression(
        solver="lbfgs",
        max_iter=2000
    )


def make_knn_model(n_neighbors=5):
    return KNeighborsClassifier(
        n_neighbors=n_neighbors,
        metric="cosine",
        weights="distance",
        n_jobs = -1
    )


# ============================================================
# Logistic OOF main model
# ============================================================

def fit_logistic_oof_with_threshold(
    y_train_array,
    X_train_conditioned,
    X_test_conditioned,
    beta=1.0,
    max_cv_splits=5
):
    result = {}

    if np.unique(y_train_array).size == 1:
        constant_value = int(y_train_array[0])
        constant_score = 8.0 if constant_value == 1 else -8.0

        result["threshold"] = 0.0
        result["oof_train_score"] = np.full(X_train_conditioned.shape[0], constant_score, dtype=float)
        result["oof_train_prob"] = np.full(X_train_conditioned.shape[0], float(constant_value), dtype=float)

        result["full_train_score"] = np.full(X_train_conditioned.shape[0], constant_score, dtype=float)
        result["full_train_prob"] = np.full(X_train_conditioned.shape[0], float(constant_value), dtype=float)

        result["test_score"] = np.full(X_test_conditioned.shape[0], constant_score, dtype=float)
        result["test_prob"] = np.full(X_test_conditioned.shape[0], float(constant_value), dtype=float)

        result["model_type"] = "constant"
        result["model"] = None
        return result

    cv_splits = choose_cv_splits(y_train_array, max_cv_splits)

    if cv_splits >= 2:
        skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=15458)
        oof_train_score = np.zeros(X_train_conditioned.shape[0], dtype=float)
        oof_train_prob = np.zeros(X_train_conditioned.shape[0], dtype=float)

        for train_index, val_index in skf.split(np.zeros(X_train_conditioned.shape[0]), y_train_array):
            fold_model = make_logit_model()
            fold_model.fit(X_train_conditioned[train_index], y_train_array[train_index])

            oof_train_score[val_index] = fold_model.decision_function(X_train_conditioned[val_index])
            oof_train_prob[val_index] = fold_model.predict_proba(X_train_conditioned[val_index])[:, 1]

        threshold = tune_scut_threshold(oof_train_score, y_train_array, beta=beta)
    else:
        fallback_model = make_logit_model()
        fallback_model.fit(X_train_conditioned, y_train_array)
        oof_train_score = fallback_model.decision_function(X_train_conditioned)
        oof_train_prob = fallback_model.predict_proba(X_train_conditioned)[:, 1]
        threshold = tune_scut_threshold(oof_train_score, y_train_array, beta=beta)

    full_model = make_logit_model()
    full_model.fit(X_train_conditioned, y_train_array)

    full_train_score = full_model.decision_function(X_train_conditioned)
    full_train_prob = full_model.predict_proba(X_train_conditioned)[:, 1]

    test_score = full_model.decision_function(X_test_conditioned)
    test_prob = full_model.predict_proba(X_test_conditioned)[:, 1]

    result["threshold"] = float(threshold)
    result["oof_train_score"] = oof_train_score
    result["oof_train_prob"] = oof_train_prob

    result["full_train_score"] = full_train_score
    result["full_train_prob"] = full_train_prob

    result["test_score"] = test_score
    result["test_prob"] = test_prob

    result["model_type"] = "conditioned_logit"
    result["model"] = full_model
    return result


# ============================================================
# KNN fallback for low-support labels
# ============================================================

def fit_knn_full(y_train_array, X_train_conditioned, X_test_conditioned, n_neighbors=5):
    """
    KNN fallback is only used for low-support labels.
    We fit on full training data and use it only in the gray zone.
    """
    result = {}

    if np.unique(y_train_array).size == 1:
        constant_value = int(y_train_array[0])
        result["train_pred"] = np.full(X_train_conditioned.shape[0], constant_value, dtype=int)
        result["test_pred"] = np.full(X_test_conditioned.shape[0], constant_value, dtype=int)
        result["train_prob"] = np.full(X_train_conditioned.shape[0], float(constant_value), dtype=float)
        result["test_prob"] = np.full(X_test_conditioned.shape[0], float(constant_value), dtype=float)
        result["model"] = None
        return result

    # Ensure n_neighbors is feasible
    n_neighbors_eff = min(n_neighbors, X_train_conditioned.shape[0])
    n_neighbors_eff = max(1, n_neighbors_eff)

    knn_model = make_knn_model(n_neighbors=n_neighbors_eff)
    knn_model.fit(X_train_conditioned, y_train_array)

    train_pred = knn_model.predict(X_train_conditioned)
    test_pred = knn_model.predict(X_test_conditioned)

    train_prob = knn_model.predict_proba(X_train_conditioned)[:, 1]
    test_prob = knn_model.predict_proba(X_test_conditioned)[:, 1]

    result["train_pred"] = train_pred.astype(int)
    result["test_pred"] = test_pred.astype(int)
    result["train_prob"] = train_prob
    result["test_prob"] = test_prob
    result["model"] = knn_model
    return result


# ============================================================
# One-label fit: logistic main + KNN fallback for low support
# ============================================================

def fit_conditioned_label_oof(level, label, y_train_label, X_train_conditioned, X_test_conditioned):
    y_train_array = y_train_label.to_numpy().astype(int)
    positive_support = int(y_train_array.sum())
    low_support = positive_support < low_support_cutoff
    threshold_beta = get_threshold_beta(level)

    logit_fit = fit_logistic_oof_with_threshold(
        y_train_array=y_train_array,
        X_train_conditioned=X_train_conditioned,
        X_test_conditioned=X_test_conditioned,
        beta=threshold_beta,
        max_cv_splits=max_cv_splits
    )

    threshold = logit_fit["threshold"]

    # Main predictions from logistic
    train_pred = (logit_fit["full_train_score"] >= threshold).astype(int)
    test_pred = (logit_fit["test_score"] >= threshold).astype(int)

    train_prob = logit_fit["full_train_prob"].copy()
    test_prob = logit_fit["test_prob"].copy()

    knn_fallback_train_count = 0
    knn_fallback_test_count = 0

    if low_support and logit_fit["model_type"] != "constant":
        knn_fit = fit_knn_full(
            y_train_array=y_train_array,
            X_train_conditioned=X_train_conditioned,
            X_test_conditioned=X_test_conditioned,
            n_neighbors=knn_neighbors
        )

        uncertain_train = np.abs(logit_fit["full_train_score"] - threshold) <= low_support_margin
        uncertain_test = np.abs(logit_fit["test_score"] - threshold) <= low_support_margin

        # For rare labels, use KNN as activator in gray zone
        train_pred[uncertain_train] = knn_fit["train_pred"][uncertain_train]
        test_pred[uncertain_test] = knn_fit["test_pred"][uncertain_test]

        train_prob[uncertain_train] = knn_fit["train_prob"][uncertain_train]
        test_prob[uncertain_test] = knn_fit["test_prob"][uncertain_test]

        knn_fallback_train_count = int(uncertain_train.sum())
        knn_fallback_test_count = int(uncertain_test.sum())

    result = {
        "label": label,
        "positive_support": positive_support,
        "low_support": low_support,
        "model_type": logit_fit["model_type"],
        "threshold": float(threshold),
        "knn_fallback_train_count": knn_fallback_train_count,
        "knn_fallback_test_count": knn_fallback_test_count,

        # OOF train outputs for downstream stacking
        "oof_train_score": logit_fit["oof_train_score"],
        "oof_train_prob": logit_fit["oof_train_prob"],

        # full-fit train/test outputs for analysis and final prediction
        "train_score": logit_fit["full_train_score"],
        "test_score": logit_fit["test_score"],
        "train_prob": train_prob,
        "test_prob": test_prob,
        "train_pred": train_pred.astype(int),
        "test_pred": test_pred.astype(int)
    }

    return result


# ============================================================
# Full OOF soft top-down training
# ============================================================

conditioned_oof_train_scores = {}
conditioned_oof_train_probs = {}

conditioned_train_scores = {}
conditioned_test_scores = {}

conditioned_train_probs = {}
conditioned_test_probs = {}

conditioned_train_pass = {}
conditioned_test_pass = {}

for level in levels:
    Y_train_level = Y_train[level].loc[X_train["id"]].copy()
    Y_test_level = Y_test[level].loc[X_test["id"]].copy()

    Y_train_level.columns = [normalize_label(column) for column in Y_train_level.columns]
    Y_test_level.columns = [normalize_label(column) for column in Y_test_level.columns]

    Y_train_level = Y_train_level.loc[:, ~Y_train_level.columns.duplicated()]
    Y_test_level = Y_test_level.loc[:, ~Y_test_level.columns.duplicated()]

    Y_train_level = Y_train_level.reindex(columns=all_level_labels[level], fill_value=0)
    Y_test_level = Y_test_level.reindex(columns=all_level_labels[level], fill_value=0)

    level_oof_train_scores = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=float)
    level_oof_train_probs = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=float)

    level_train_scores = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=float)
    level_test_scores = pd.DataFrame(index=X_test["id"], columns=Y_train_level.columns, dtype=float)

    level_train_probs = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=float)
    level_test_probs = pd.DataFrame(index=X_test["id"], columns=Y_train_level.columns, dtype=float)

    level_train_pass = pd.DataFrame(index=X_train["id"], columns=Y_train_level.columns, dtype=int)
    level_test_pass = pd.DataFrame(index=X_test["id"], columns=Y_train_level.columns, dtype=int)

    for label in Y_train_level.columns:
        if level == "h1":
            X_train_conditioned = X_train_vec
            X_test_conditioned = X_test_vec
            parent_label = None
        else:
            parent_label = level_parent_lookup[level][label]

            # TRAIN SIDE: use OOF parent outputs only
            parent_train_scores = conditioned_oof_train_scores[parent_level[level]][parent_label].to_numpy()
            parent_train_probs = conditioned_oof_train_probs[parent_level[level]][parent_label].to_numpy()

            # TEST SIDE: use full-fit parent test outputs
            parent_test_scores = conditioned_test_scores[parent_level[level]][parent_label].to_numpy()
            parent_test_probs = conditioned_test_probs[parent_level[level]][parent_label].to_numpy()

            X_train_conditioned = build_conditioned_matrix(
                X_train_vec,
                parent_scores=parent_train_scores,
                parent_probs=parent_train_probs
            )

            X_test_conditioned = build_conditioned_matrix(
                X_test_vec,
                parent_scores=parent_test_scores,
                parent_probs=parent_test_probs
            )

        fitted_label = fit_conditioned_label_oof(
            level=level,
            label=label,
            y_train_label=Y_train_level[label],
            X_train_conditioned=X_train_conditioned,
            X_test_conditioned=X_test_conditioned
        )

        level_oof_train_scores[label] = fitted_label["oof_train_score"]
        level_oof_train_probs[label] = fitted_label["oof_train_prob"]

        level_train_scores[label] = fitted_label["train_score"]
        level_test_scores[label] = fitted_label["test_score"]

        level_train_probs[label] = fitted_label["train_prob"]
        level_test_probs[label] = fitted_label["test_prob"]

        level_train_pass[label] = fitted_label["train_pred"]
        level_test_pass[label] = fitted_label["test_pred"]

        conditioning_results.append({
            "level": level,
            "label": label,
            "parent_label": parent_label,
            "positive_support": fitted_label["positive_support"],
            "low_support": fitted_label["low_support"],
            "threshold": fitted_label["threshold"],
            "model_type": fitted_label["model_type"],
            "knn_fallback_train_count": fitted_label["knn_fallback_train_count"],
            "knn_fallback_test_count": fitted_label["knn_fallback_test_count"]
        })

    conditioned_oof_train_scores[level] = level_oof_train_scores
    conditioned_oof_train_probs[level] = level_oof_train_probs

    conditioned_train_scores[level] = level_train_scores
    conditioned_test_scores[level] = level_test_scores

    conditioned_train_probs[level] = level_train_probs
    conditioned_test_probs[level] = level_test_probs

    conditioned_train_pass[level] = level_train_pass.astype(int)
    conditioned_test_pass[level] = level_test_pass.astype(int)


# ============================================================
# Final assembled prediction tables
# ============================================================

soft_topdown_predictions = pd.concat(
    [conditioned_test_pass[level] for level in levels],
    axis=1
)
soft_topdown_predictions = soft_topdown_predictions.loc[:, ~soft_topdown_predictions.columns.duplicated()]
soft_topdown_predictions = soft_topdown_predictions.reindex(sorted(soft_topdown_predictions.columns), axis=1)

soft_topdown_confidence = pd.concat(
    [conditioned_test_probs[level] for level in levels],
    axis=1
)
soft_topdown_confidence = soft_topdown_confidence.loc[:, ~soft_topdown_confidence.columns.duplicated()]
soft_topdown_confidence = soft_topdown_confidence.reindex(columns=soft_topdown_predictions.columns)

# Optional hierarchy closure
for child, parent in normalized_parent_map.items():
    if child in soft_topdown_predictions.columns and parent in soft_topdown_predictions.columns:
        soft_topdown_predictions[parent] = np.where(
            soft_topdown_predictions[child] == 1,
            1,
            soft_topdown_predictions[parent]
        )

soft_topdown_true = pd.concat(
    [Y_test[level].loc[X_test["id"]].rename(columns=normalize_label) for level in levels],
    axis=1
)
soft_topdown_true = soft_topdown_true.loc[:, ~soft_topdown_true.columns.duplicated()]
soft_topdown_true = soft_topdown_true.reindex(columns=soft_topdown_predictions.columns, fill_value=0)

conditioning_results_df = pd.DataFrame(conditioning_results)


# ============================================================
# Reporting
# ============================================================

print("=" * 80)
print("FULL OOF SOFT TOP-DOWN CONDITIONING MODEL WITH LOW-SUPPORT KNN FALLBACK")
print("=" * 80)
print(f"Total labels trained: {conditioning_results_df.shape[0]}")
print(f"Low-support cutoff: {low_support_cutoff}")
print(f"Labels marked low-support: {int(conditioning_results_df['low_support'].sum())}")
print(f"Total train predictions touched by KNN fallback: {conditioning_results_df['knn_fallback_train_count'].sum()}")
print(f"Total test predictions touched by KNN fallback: {conditioning_results_df['knn_fallback_test_count'].sum()}")

print("\nOverall Metrics")
print(f"Micro precision: {precision_score(soft_topdown_true, soft_topdown_predictions, average='micro', zero_division=0):.4f}")
print(f"Micro recall: {recall_score(soft_topdown_true, soft_topdown_predictions, average='micro', zero_division=0):.4f}")
print(f"Micro F1: {f1_score(soft_topdown_true, soft_topdown_predictions, average='micro', zero_division=0):.4f}")
print(f"Macro precision: {precision_score(soft_topdown_true, soft_topdown_predictions, average='macro', zero_division=0):.4f}")
print(f"Macro recall: {recall_score(soft_topdown_true, soft_topdown_predictions, average='macro', zero_division=0):.4f}")
print(f"Macro F1: {f1_score(soft_topdown_true, soft_topdown_predictions, average='macro', zero_division=0):.4f}")

print("\nPer-level Metrics")
for level in levels:
    level_true = Y_test[level].loc[X_test["id"]].rename(columns=normalize_label)
    level_true = level_true.loc[:, ~level_true.columns.duplicated()]
    level_true = level_true.reindex(columns=all_level_labels[level], fill_value=0)

    level_pred = conditioned_test_pass[level].loc[level_true.index, level_true.columns]

    print(
        f"{level}: "
        f"precision = {precision_score(level_true, level_pred, average='micro', zero_division=0):.4f}, "
        f"recall = {recall_score(level_true, level_pred, average='micro', zero_division=0):.4f}, "
        f"F1 = {f1_score(level_true, level_pred, average='micro', zero_division=0):.4f}"
    )

FULL OOF SOFT TOP-DOWN CONDITIONING MODEL WITH LOW-SUPPORT KNN FALLBACK
Total labels trained: 116
Low-support cutoff: 250
Labels marked low-support: 67
Total train predictions touched by KNN fallback: 3266
Total test predictions touched by KNN fallback: 62366

Overall Metrics
Micro precision: 0.5227
Micro recall: 0.7790
Micro F1: 0.6256
Macro precision: 0.3748
Macro recall: 0.6335
Macro F1: 0.4025

Per-level Metrics
h1: precision = 0.9100, recall = 0.8988, F1 = 0.9044
h2: precision = 0.4565, recall = 0.7554, F1 = 0.5691
h3: precision = 0.6890, recall = 0.6547, F1 = 0.6714
h4: precision = 0.4357, recall = 0.7269, F1 = 0.5449
h5: precision = 0.9738, recall = 0.9738, F1 = 0.9738


V5: Flat Logistic + NB Ensemble Meta Model
- No hierarchy in labels like V1
- Instead of V1 low support fallback for NB usage, instead trains a "meta" model learning hwo to combine Logistic and NB per label
- Constant label shortcut - if a label is always 0 or 1 in train, just stores constant probability
- Trains Logistic and NB model on each fold in cross-validation, then trains Logistic "meta" model fitting weights on those two probability signals
- Per label threshold tuning

In [65]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report


# ============================================================
# 1. CREATE FLAT LABEL MATRICES
# ============================================================

levels = ["h1", "h2", "h3", "h4", "h5"]

Y_train_flat = pd.concat((Y_train[level] for level in levels), axis=1)
Y_train_flat = Y_train_flat.loc[X_train["id"]]
Y_train_flat = Y_train_flat.reindex(sorted(Y_train_flat.columns), axis=1)
print("Y_train_flat shape:", Y_train_flat.shape)

Y_test_flat = pd.concat((Y_test[level] for level in levels), axis=1)
Y_test_flat = Y_test_flat.loc[X_test["id"]]
Y_test_flat = Y_test_flat.reindex(sorted(Y_test_flat.columns), axis=1)
print("Y_test_flat shape:", Y_test_flat.shape)


# ============================================================
# 2. HELPERS
# ============================================================

random_state = 15458
max_cv_splits = 5

def choose_cv_splits(y, max_splits=5):
    positives = int(y.sum())
    negatives = int(len(y) - positives)
    return min(max_splits, positives, negatives)

def tune_threshold_from_probs(probs, y_true, grid_size=101):
    """
    SCut-style threshold tuning in probability space.
    Optimizes F1 for this label on OOF meta probabilities.
    """
    probs = np.asarray(probs, dtype=float)
    y_true = np.asarray(y_true).astype(int)

    if np.unique(y_true).size == 1:
        # constant label
        return 0.5 if y_true[0] == 1 else 1.0

    # robust grid based on observed probability range
    lower, upper = np.quantile(probs, [0.01, 0.99])
    lower = float(max(0.0, lower))
    upper = float(min(1.0, upper))

    if lower == upper:
        lower, upper = 0.0, 1.0

    threshold_grid = np.linspace(lower, upper, grid_size)
    best_threshold = 0.5
    best_score = -1.0

    for threshold in threshold_grid:
        y_pred = (probs >= threshold).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)

    return best_threshold


# ============================================================
# 3. STORAGE FOR OOF BASE PROBS, FULL BASE MODELS, META MODELS
# ============================================================

labels = list(Y_train_flat.columns)
n_train = X_train_vec.shape[0]
n_test = X_test_vec.shape[0]
n_labels = len(labels)

# OOF train probabilities from base models
logit_oof_probs = np.zeros((n_train, n_labels), dtype=float)
nb_oof_probs = np.zeros((n_train, n_labels), dtype=float)

# full-fit test probabilities from base models
logit_test_probs = np.zeros((n_test, n_labels), dtype=float)
nb_test_probs = np.zeros((n_test, n_labels), dtype=float)

# also keep full-fit train probs if you want to inspect them later
logit_train_probs = np.zeros((n_train, n_labels), dtype=float)
nb_train_probs = np.zeros((n_train, n_labels), dtype=float)

# metadata
meta_models = {}
label_thresholds = {}
label_diagnostics = []


# ============================================================
# 4. FIT PER LABEL:
#    BASE LOGISTIC OOF + BASE NB OOF + META COMBINER
# ============================================================

for j, label in enumerate(labels):
    y = Y_train_flat.iloc[:, j].to_numpy().astype(int)

    positives = int(y.sum())
    negatives = int(len(y) - positives)

    print(f"Fitting label {j+1}/{n_labels}: {label} | support={positives}")

    # --------------------------------------------------------
    # constant-label case
    # --------------------------------------------------------
    if np.unique(y).size == 1:
        constant_value = int(y[0])

        logit_oof_probs[:, j] = float(constant_value)
        nb_oof_probs[:, j] = float(constant_value)

        logit_train_probs[:, j] = float(constant_value)
        nb_train_probs[:, j] = float(constant_value)

        logit_test_probs[:, j] = float(constant_value)
        nb_test_probs[:, j] = float(constant_value)

        meta_models[label] = {
            "type": "constant",
            "value": constant_value
        }
        label_thresholds[label] = 0.5 if constant_value == 1 else 1.0

        label_diagnostics.append({
            "label": label,
            "support": positives,
            "meta_type": "constant",
            "threshold": label_thresholds[label]
        })
        continue

    # --------------------------------------------------------
    # build OOF probs for logistic and NB
    # --------------------------------------------------------
    cv_splits = choose_cv_splits(y, max_splits=max_cv_splits)

    if cv_splits >= 2:
        skf = StratifiedKFold(
            n_splits=cv_splits,
            shuffle=True,
            random_state=random_state
        )

        for train_idx, val_idx in skf.split(np.zeros(n_train), y):
            X_train_fold_vec = X_train_vec[train_idx]
            X_val_fold_vec = X_train_vec[val_idx]

            X_train_fold_count = X_train_count[train_idx]
            X_val_fold_count = X_train_count[val_idx]

            y_train_fold = y[train_idx]

            # base logistic
            logit_fold = LogisticRegression(
                solver="lbfgs",
                max_iter=2000
            )
            logit_fold.fit(X_train_fold_vec, y_train_fold)
            logit_oof_probs[val_idx, j] = logit_fold.predict_proba(X_val_fold_vec)[:, 1]

            # base NB
            nb_fold = MultinomialNB(alpha=1.0)
            nb_fold.fit(X_train_fold_count, y_train_fold)
            nb_oof_probs[val_idx, j] = nb_fold.predict_proba(X_val_fold_count)[:, 1]

    else:
        # fallback if too few positives/negatives for CV
        logit_fold = LogisticRegression(
            solver="lbfgs",
            max_iter=2000
        )
        logit_fold.fit(X_train_vec, y)
        logit_oof_probs[:, j] = logit_fold.predict_proba(X_train_vec)[:, 1]

        nb_fold = MultinomialNB(alpha=1.0)
        nb_fold.fit(X_train_count, y)
        nb_oof_probs[:, j] = nb_fold.predict_proba(X_train_count)[:, 1]

    # --------------------------------------------------------
    # fit full base models for final train/test probabilities
    # --------------------------------------------------------
    logit_full = LogisticRegression(
        solver="lbfgs",
        max_iter=2000
    )
    logit_full.fit(X_train_vec, y)

    nb_full = MultinomialNB(alpha=1.0)
    nb_full.fit(X_train_count, y)

    logit_train_probs[:, j] = logit_full.predict_proba(X_train_vec)[:, 1]
    nb_train_probs[:, j] = nb_full.predict_proba(X_train_count)[:, 1]

    logit_test_probs[:, j] = logit_full.predict_proba(X_test_vec)[:, 1]
    nb_test_probs[:, j] = nb_full.predict_proba(X_test_count)[:, 1]

    # --------------------------------------------------------
    # meta-model training data from OOF base probs
    # --------------------------------------------------------
    X_meta_train = np.column_stack([
        logit_oof_probs[:, j],
        nb_oof_probs[:, j]
    ])

    X_meta_test = np.column_stack([
        logit_test_probs[:, j],
        nb_test_probs[:, j]
    ])

    # --------------------------------------------------------
    # fit meta combiner
    # --------------------------------------------------------
    if np.unique(y).size == 1:
        meta_models[label] = {
            "type": "constant",
            "value": int(y[0])
        }
        meta_oof_probs = np.full(n_train, float(y[0]), dtype=float)
        meta_test_probs = np.full(n_test, float(y[0]), dtype=float)

    else:
        meta_model = LogisticRegression(
            solver="lbfgs",
            max_iter=2000
        )
        meta_model.fit(X_meta_train, y)

        meta_oof_probs = meta_model.predict_proba(X_meta_train)[:, 1]
        meta_test_probs = meta_model.predict_proba(X_meta_test)[:, 1]

        meta_models[label] = {
            "type": "logistic",
            "model": meta_model
        }

    # --------------------------------------------------------
    # tune per-label threshold on OOF meta probs
    # --------------------------------------------------------
    threshold = tune_threshold_from_probs(meta_oof_probs, y, grid_size=101)
    label_thresholds[label] = threshold

    # temporarily store test meta probs inside diagnostics table
    label_diagnostics.append({
        "label": label,
        "support": positives,
        "meta_type": meta_models[label]["type"],
        "threshold": threshold
    })


# ============================================================
# 5. REBUILD FINAL META TRAIN / TEST PROB MATRICES
# ============================================================

meta_train_probs = np.zeros((n_train, n_labels), dtype=float)
meta_test_probs = np.zeros((n_test, n_labels), dtype=float)

for j, label in enumerate(labels):
    y = Y_train_flat.iloc[:, j].to_numpy().astype(int)

    if meta_models[label]["type"] == "constant":
        constant_value = meta_models[label]["value"]
        meta_train_probs[:, j] = float(constant_value)
        meta_test_probs[:, j] = float(constant_value)
    else:
        X_meta_train = np.column_stack([
            logit_train_probs[:, j],
            nb_train_probs[:, j]
        ])
        X_meta_test = np.column_stack([
            logit_test_probs[:, j],
            nb_test_probs[:, j]
        ])

        meta_model = meta_models[label]["model"]
        meta_train_probs[:, j] = meta_model.predict_proba(X_meta_train)[:, 1]
        meta_test_probs[:, j] = meta_model.predict_proba(X_meta_test)[:, 1]


# ============================================================
# 6. APPLY META THRESHOLDS
# ============================================================

thresholds = np.array([label_thresholds[label] for label in labels], dtype=float)

Y_pred_flat = (meta_test_probs >= thresholds[np.newaxis, :]).astype(int)
Y_pred_train_flat = (meta_train_probs >= thresholds[np.newaxis, :]).astype(int)


# ============================================================
# 7. OPTIONAL: ENFORCE PARENT CONSISTENCY
# ============================================================

enforce_hierarchy = True

if enforce_hierarchy:
    parent_map = {}
    for child, parent in zip(h2["h2"].astype(str), h2["h1"].astype(str)):
        parent_map[child] = parent
    for child, parent in zip(h3["h3"].astype(str), h3["h2"].astype(str)):
        parent_map[child] = parent
    for child, parent in zip(h4["h4"].astype(str), h4["h3"].astype(str)):
        parent_map[child] = parent
    for child, parent in zip(h5["h5"].astype(str), h5["h4"].astype(str)):
        parent_map[child] = parent

    predictions_df = pd.DataFrame(
        Y_pred_flat,
        columns=labels,
        index=X_test["id"]
    )

    for child, parent in parent_map.items():
        if child in predictions_df.columns and parent in predictions_df.columns:
            predictions_df[parent] = np.where(
                predictions_df[child] == 1,
                1,
                predictions_df[parent]
            )

    Y_pred_flat = predictions_df.to_numpy()

else:
    predictions_df = pd.DataFrame(
        Y_pred_flat,
        columns=labels,
        index=X_test["id"]
    )


# ============================================================
# 8. CONFIDENCE SCORES
# ============================================================

# These are now meta-model probabilities, already in [0, 1]
confidence_scores = meta_test_probs
confidence_df = pd.DataFrame(
    confidence_scores,
    columns=labels,
    index=X_test["id"]
)


# ============================================================
# 9. REPORTING
# ============================================================

print("=" * 80)
print("STACKED OOF COMBINER RESULTS")
print("=" * 80)

print(classification_report(
    Y_test_flat,
    Y_pred_flat,
    zero_division=0
))

diagnostics_df = pd.DataFrame(label_diagnostics).sort_values(
    by=["support", "label"],
    ascending=[True, True]
)

print("\nSmall-support labels and thresholds:")
print(diagnostics_df.head(25))


# ============================================================
# 10. OPTIONAL: INSPECT WHAT META MODEL LEARNED
# ============================================================

meta_weight_rows = []

for label in labels:
    if meta_models[label]["type"] == "logistic":
        coef = meta_models[label]["model"].coef_[0]
        intercept = meta_models[label]["model"].intercept_[0]
        meta_weight_rows.append({
            "label": label,
            "coef_logit_prob": coef[0],
            "coef_nb_prob": coef[1],
            "intercept": intercept,
            "threshold": label_thresholds[label],
            "support": int(Y_train_flat[label].sum())
        })
    else:
        meta_weight_rows.append({
            "label": label,
            "coef_logit_prob": np.nan,
            "coef_nb_prob": np.nan,
            "intercept": np.nan,
            "threshold": label_thresholds[label],
            "support": int(Y_train_flat[label].sum())
        })

meta_weights_df = pd.DataFrame(meta_weight_rows).sort_values(
    by="support",
    ascending=True
)

print("\nMeta-model weights preview:")
print(meta_weights_df.head(25))

Y_train_flat shape: (23149, 103)
Y_test_flat shape: (390616, 103)
Fitting label 1/103: C11        | support=674
Fitting label 2/103: C12        | support=381
Fitting label 3/103: C13        | support=947
Fitting label 4/103: C14        | support=160
Fitting label 5/103: C15        | support=4179
Fitting label 6/103: C151       | support=2366
Fitting label 7/103: C1511      | support=399
Fitting label 8/103: C152       | support=1930
Fitting label 9/103: C16        | support=49
Fitting label 10/103: C17        | support=1172
Fitting label 11/103: C171       | support=437
Fitting label 12/103: C172       | support=285
Fitting label 13/103: C173       | support=76
Fitting label 14/103: C174       | support=246
Fitting label 15/103: C18        | support=1462
Fitting label 16/103: C181       | support=1205
Fitting label 17/103: C182       | support=142
Fitting label 18/103: C183       | support=202
Fitting label 19/103: C21        | support=793
Fitting label 20/103: C22        | support=190

V6: Level-Specific Logistic and NB Ensemble Model
- Separates meta model training from V5 into specific hierarchy levels
- Adds an interaction term to train the model on how much logistic and NB models agree in their scores

In [75]:
#Import Packages to keep this section independent
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, fbeta_score, precision_score, recall_score, f1_score

In [78]:
#Global variables for rest of this model
levels = ["h1", "h2", "h3", "h4", "h5"]
random_state = 15458
max_cv_splits = 5

meta_min_support = 100 #less than 100 positive support in label, don't stack ensemble model but do base model

thresh_beta_default = 0.5 #how much the model favors precision
thresh_floor_default = 0.05 #prevents threshold from going to 0
thresh_grid_size = 101

#Per level configuration settings - minimum support and threshold should go down as training examples get smaller, but keep focus on precision in the model
level_configs = {
    "h1": {"beta": 1.0, "thresh_floor": 0.10, "meta_min_support": 125},
    "h2": {"beta": 0.5, "thresh_floor": 0.05, "meta_min_support": 75},
    "h3": {"beta": 0.5, "thresh_floor": 0.05, "meta_min_support": 50},
    "h4": {"beta": 0.5, "thresh_floor": 0.05, "meta_min_support": 30},
    "h5": {"beta": 0.5, "thresh_floor": 0.05, "meta_min_support": 20},
}


In [81]:
#Helper Functions

#Choose the CV splits to balance amount of positives and negatives CHECK
def choose_cv_splits(y, max_splits = 5):
    positives = int(y.sum()) #y is matrix of 0s and 1s across articles and labels
    negatives = int(len(y) - positives)
    return min(max_splits, positives, negatives)

#Tune the threshold using F-beta (units of probability space), applying floor to avoid thresholds close to 0
def tune_threshold_from_probs(probs, y_true, beta = 0.5, thresh_floor = 0.05, grid_size = 101):
    '''
    Inputs:
        probability scores (0-1) for one label, true binary classifications for one label (0/1), beta, threshold floor, grid size of thresholds to test
    '''
    probs = np.asarray(probs, dtype = float)
    y_true = np.asarray(y_true).astype(int)

    if np.unique(y_true).size == 1: #if true binary classifications for one label are all 0s or 1s, assign threshold to 0.5 if true (might not always be true) or 1 if false (avoid FP and FN)
        return 0.5 if y_true[0] == 1 else 1.0

    thresh_grid = np.linspace(0.0, 1.0, grid_size)

    #initialize values
    best_thresh = thresh_floor
    best_score = -1.0

    for thresh in thresh_grid:
        y_pred = (probs >= thresh).astype(int) #see how predicted ys with threshold score
        score = fbeta_score(y_true, y_pred, beta=beta, zero_division = 0)
        if score > best_score:
            best_score = score
            best_thresh = float(thresh)

    return max(best_thresh, thresh_floor)

#Handle edge cases so model always returns probability vector P(y = 1) of length n_samples
def safe_predict_proba_binary(model, X):
    #Edge Case 1: Model only learned one class (all 0s probabilities or all 1s probabilities)
    if hasattr(model, "classes_") and len(model.classes_) == 1:
        only_class = model.classes_[0]
        if only_class == 1:
            return np.ones(X.shape[0], dtype = float) #probability vector of length of number of articles trained
        return np.zeros(X.shape[0], dtype = float)

    #Edge Case 2: Model probabilities should give matrix of shape (prob y = 0, prob y = 1) for each article
    probs = model.predict_proba(X)
    if probs.ndim == 1:
        return probs.astype(float)
    return probs[:, 1].astype(float) #return probability of Y = 1

#Clean summary helper function
def summarize_level(y_true_df, y_pred_df):
    y_true = y_true_df.to_numpy().astype(int) #to be safe, avoid data type surprises in dfs and strips index/column labels
    y_pred = y_pred_df.to_numpy().astype(int)

    return {
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [88]:
#Storing results here
level_results = {}

#FOR EACH HIERARCHY LEVEL
for level in levels:
    #Nice formatting from ChatGPT
    print("=" * 90)
    print(f"LEVEL: {level}")
    print("=" * 90)

    #Data checking/cleaning like above, putting here so model stands alone from previous versions
    Y_train_level = Y_train[level].loc[X_train["id"]].copy()
    Y_test_level = Y_test[level].loc[X_test["id"]].copy()

    Y_train_level = Y_train_level.reindex(sorted(Y_train_level.columns), axis = 1)
    Y_test_level = Y_test_level.reindex(Y_train_level.columns, axis = 1)

    labels = list(Y_train_level.columns)
    n_labels = len(labels)
    n_train = X_train_vec.shape[0]
    n_test = X_test_vec.shape[0]

    print(f"Train shape: {Y_train_level.shape}")
    print(f"Test shape:  {Y_test_level.shape}")

    #Get the base configurations per level
    beta = level_configs.get(level, {}).get("beta", thresh_beta_default) #extra safe, shouldn't hit defaults
    thresh_floor = level_configs.get(level, {}).get("thresh_floor", thresh_floor_default)
    level_meta_min_support = level_configs.get(level, {}).get("meta_min_support", meta_min_support)

    #Create storage for the "Out of Fold" OOF probabilities for logistic and NB models from CV training
    logit_oof_probs = np.zeros((n_train, n_labels), dtype = float)
    logit_train_probs = np.zeros((n_train, n_labels), dtype=float)
    logit_test_probs = np.zeros((n_test, n_labels), dtype=float)

    nb_oof_probs = np.zeros((n_train, n_labels), dtype = float)
    nb_train_probs = np.zeros((n_train, n_labels), dtype=float)
    nb_test_probs = np.zeros((n_test, n_labels), dtype=float)

    meta_train_probs = np.zeros((n_train, n_labels), dtype=float)
    meta_test_probs = np.zeros((n_test, n_labels), dtype=float)

    #Store label-tuned thresholds, meta models and weights and other diagnostics
    label_thresholds = {}
    meta_models = {}
    label_diagnostics = []
    meta_weights = []

    #FOR EACH LABEL IN THE LEVEL
    for j, label in enumerate(labels):
        y = Y_train_level.iloc[:, j].to_numpy().astype(int)
        support = int(y.sum())

        print(f"Fitting {level} label {j}/{n_labels - 1}: {label} | support={support}")

        #only one classification for the label (all 1 or all 0, then skip the model and assign all probabilities to that value)
        if np.unique(y).size == 1:
            constant_value = int(y[0])

            logit_oof_probs[:, j] = float(constant_value)
            nb_oof_probs[:, j] = float(constant_value)

            logit_train_probs[:, j] = float(constant_value)
            nb_train_probs[:, j] = float(constant_value)

            logit_test_probs[:, j] = float(constant_value)
            nb_test_probs[:, j] = float(constant_value)

            meta_train_probs[:, j] = float(constant_value)
            meta_test_probs[:, j] = float(constant_value)

            threshold = 0.5 if constant_value == 1 else 1.0
            label_thresholds[label] = threshold
            meta_models[label] = {"type": "constant", "value": constant_value}

            label_diagnostics.append({
                "label": label,
                "support": support,
                "meta_type": "constant",
                "threshold": threshold
            })

            meta_weights.append({
                "label": label,
                "support": support,
                "meta_type": "constant",
                "coef_logit_prob": np.nan,
                "coef_nb_prob": np.nan,
                "coef_interaction": np.nan,
                "intercept": np.nan,
                "threshold": threshold
            })

            continue

        #otherwise go to the model - start with cross validation trained models
        cv_splits = choose_cv_splits(y, max_splits = max_cv_splits)

        if cv_splits >= 2:
            skf = StratifiedKFold(
                n_splits = cv_splits,
                shuffle = True,
                random_state = random_state
            )

            for train_idx, val_idx in skf.split(np.zeros(n_train), y):
                X_train_fold_vec = X_train_vec[train_idx]
                X_val_fold_vec = X_train_vec[val_idx]

                X_train_fold_count = X_train_count[train_idx]
                X_val_fold_count = X_train_count[val_idx]

                Y_train_fold = y[train_idx]

                #Base Logistic Regression Model
                logit_fold = LogisticRegression(
                    solver = "lbfgs",
                    max_iter = 2000,
                    class_weight = "balanced"
                )
                logit_fold.fit(X_train_fold_vec, Y_train_fold)
                logit_oof_probs[val_idx, j] = safe_predict_proba_binary(logit_fold, X_val_fold_vec)

                #Base NB Regression Model
                nb_fold = MultinomialNB(alpha = 1.0)
                nb_fold.fit(X_train_fold_count, Y_train_fold)
                nb_oof_probs[val_idx, j] = safe_predict_proba_binary(nb_fold, X_val_fold_count)

        else: #no stratified cross-validation
            logit_fold = LogisticRegression(
                solver = "lbfgs",
                max_iter = 2000,
                class_weight = "balanced"
            )
            logit_fold.fit(X_train_vec, y)
            logit_oof_probs[:, j] = safe_predict_proba_binary(logit_fold, X_train_vec)

            nb_fold = MultinomialNB(alpha = 1.0)
            nb_fold.fit(X_train_count, y)
            nb_oof_probs[:, j] = safe_predict_proba_binary(nb_fold, X_train_count)

        #Full-fit base models
        logit_full = LogisticRegression(
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced"
        )
        logit_full.fit(X_train_vec, y)

        nb_full = MultinomialNB(alpha=1.0)
        nb_full.fit(X_train_count, y)

        logit_train_probs[:, j] = safe_predict_proba_binary(logit_full, X_train_vec)
        nb_train_probs[:, j] = safe_predict_proba_binary(nb_full, X_train_count)

        logit_test_probs[:, j] = safe_predict_proba_binary(logit_full, X_test_vec)
        nb_test_probs[:, j] = safe_predict_proba_binary(nb_full, X_test_count)

        # Slight NB damping to reduce overconfidence
        nb_oof_clip = np.sqrt(np.clip(nb_oof_probs[:, j], 0.0, 1.0))
        nb_test_clip = np.sqrt(np.clip(nb_test_probs[:, j], 0.0, 1.0))
        nb_train_clip = np.sqrt(np.clip(nb_train_probs[:, j], 0.0, 1.0))

        #Build out the meta features with interactions, so the model learns to incorporate NB only when it agrees with Logistic Reg
        X_meta_oof = np.column_stack([
            logit_oof_probs[:, j],
            nb_oof_clip,
            logit_oof_probs[:, j] * nb_oof_clip
        ])

        X_meta_train = np.column_stack([
            logit_train_probs[:, j],
            nb_train_clip,
            logit_train_probs[:, j] * nb_train_clip
        ])

        X_meta_test = np.column_stack([
            logit_test_probs[:, j],
            nb_test_clip,
            logit_test_probs[:, j] * nb_test_clip
        ])

        #ONTO THE META MODEL
        #Case 1: Low Support, don't use stacked features, just logistic regression
        if support < level_meta_min_support:
            meta_type = "logistic_only"

            meta_train_probs[:, j] = logit_train_probs[:, j] #GLOBAL VAR
            meta_test_probs[:, j] = logit_test_probs[:, j] #GLOBAL VAR
            meta_oof_probs = logit_oof_probs[:, j] #this is a temporary variable

            #tune the right threshold for the label
            threshold = tune_threshold_from_probs(meta_oof_probs, y, beta, thresh_floor)
            label_thresholds[label] = threshold
            meta_models[label] = {"type": meta_type}

            label_diagnostics.append({
                "label": label,
                "support": support,
                "meta_type": meta_type,
                "threshold": threshold
            })

            meta_weights.append({
                "label": label,
                "support": support,
                "meta_type": meta_type,
                "coef_logit_prob": np.nan,
                "coef_nb_prob": np.nan,
                "coef_interaction": np.nan,
                "intercept": np.nan,
                "threshold": threshold
            })
        #Case 2: Support is fine, use stacked features
        else:
            #meta model does not have balanced class - will not get pushed to recall so precision is better than previous model versions
            meta_model = LogisticRegression(
                solver = "lbfgs",
                max_iter = 2000
            )
            meta_model.fit(X_meta_oof, y) #fit on OOF train features

            meta_oof_probs = safe_predict_proba_binary(meta_model, X_meta_oof)
            meta_train_probs[:, j] = safe_predict_proba_binary(meta_model, X_meta_train)
            meta_test_probs[:, j] = safe_predict_proba_binary(meta_model, X_meta_test)

            #tune the right threshold
            threshold = tune_threshold_from_probs(meta_oof_probs, y, beta=beta, thresh_floor=thresh_floor, grid_size=thresh_grid_size)
            label_thresholds[label] = threshold
            meta_models[label] = {"type": "stacked", "model": meta_model}

            label_diagnostics.append({
                "label": label,
                "support": support,
                "meta_type": "stacked",
                "threshold": threshold
            })

            coef = meta_model.coef_[0] #vector of coefficients in front of stacked features
            intercept = meta_model.intercept_[0]

            meta_weights.append({
                "label": label,
                "support": support,
                "meta_type": "stacked",
                "coef_logit_prob": coef[0],
                "coef_nb_prob": coef[1],
                "coef_interaction": coef[2],
                "intercept": intercept,
                "threshold": threshold
            })

    #FINAL PREDICTIONS AFTER DOING META MODEL FOR EVERY LABEL IN THE HIERARCHY LEVEL
    thresholds = np.array([label_thresholds[label] for label in labels], dtype = float)

    Y_pred_level = (meta_test_probs >= thresholds[np.newaxis, :]).astype(int)
    Y_pred_train_level = (meta_train_probs >= thresholds[np.newaxis, :]).astype(int)

    predictions_df = pd.DataFrame(
        Y_pred_level,
        columns=labels,
        index=X_test["id"]
    )

    confidence_df = pd.DataFrame(
        meta_test_probs,
        columns=labels,
        index=X_test["id"]
    )

    diagnostics_df = pd.DataFrame(label_diagnostics).sort_values(
        by=["support", "label"],
        ascending=[True, True]
    )

    meta_weights_df = pd.DataFrame(meta_weights).sort_values(
        by=["support", "label"],
        ascending=[True, True]
    )

    #REPORT RESULTS FOR EACH LEVEL
    print("\nClassification report")
    print(classification_report(
        Y_test_level,
        Y_pred_level,
        zero_division=0
    ))

    metrics = summarize_level(
        Y_test_level,
        predictions_df
    )

    #Summary statistics thanks to ChatGPT
    print("Summary metrics")
    print(f"Micro precision: {metrics['micro_precision']:.4f}")
    print(f"Micro recall:    {metrics['micro_recall']:.4f}")
    print(f"Micro F1:        {metrics['micro_f1']:.4f}")
    print(f"Macro precision: {metrics['macro_precision']:.4f}")
    print(f"Macro recall:    {metrics['macro_recall']:.4f}")
    print(f"Macro F1:        {metrics['macro_f1']:.4f}")

    print("\nSmall-support labels and thresholds")
    print(diagnostics_df.head(25))

    print("\nMeta-model weights preview")
    print(meta_weights_df.head(25))

    #Storage thanks to ChatGPT
    level_results[level] = {
        "Y_train_level": Y_train_level,
        "Y_test_level": Y_test_level,
        "labels": labels,

        "logit_oof_probs": logit_oof_probs,
        "nb_oof_probs": nb_oof_probs,

        "logit_train_probs": logit_train_probs,
        "nb_train_probs": nb_train_probs,

        "logit_test_probs": logit_test_probs,
        "nb_test_probs": nb_test_probs,

        "meta_train_probs": meta_train_probs,
        "meta_test_probs": meta_test_probs,

        "thresholds": label_thresholds,
        "threshold_array": thresholds,
        "meta_models": meta_models,

        "Y_pred_test": Y_pred_level,
        "Y_pred_train": Y_pred_train_level,

        "predictions_df": predictions_df,
        "confidence_df": confidence_df,
        "diagnostics_df": diagnostics_df,
        "meta_weights_df": meta_weights_df,
        "metrics": metrics
    }

LEVEL: h1
Train shape: (23149, 4)
Test shape:  (390616, 4)
Fitting h1 label 0/3: CCAT       | support=10786
Fitting h1 label 1/3: ECAT       | support=3449
Fitting h1 label 2/3: GCAT       | support=6970
Fitting h1 label 3/3: MCAT       | support=5882

Classification report
              precision    recall  f1-score   support

           0       0.93      0.91      0.92    185137
           1       0.80      0.77      0.78     58247
           2       0.92      0.93      0.92    116136
           3       0.92      0.90      0.91     99593

   micro avg       0.91      0.90      0.90    459113
   macro avg       0.89      0.88      0.89    459113
weighted avg       0.91      0.90      0.90    459113
 samples avg       0.93      0.93      0.92    459113

Summary metrics
Micro precision: 0.9104
Micro recall:    0.8964
Micro F1:        0.9034
Macro precision: 0.8930
Macro recall:    0.8785
Macro F1:        0.8856

Small-support labels and thresholds
        label  support meta_type  thres

In [84]:
#Overall Summary thanks to ChatGPT
print("\n" + "=" * 90)
print("OVERALL SUMMARY BY LEVEL")
print("=" * 90)

overall_rows = []
for level in levels:
    m = level_results[level]["metrics"]
    overall_rows.append({
        "level": level,
        "micro_precision": m["micro_precision"],
        "micro_recall": m["micro_recall"],
        "micro_f1": m["micro_f1"],
        "macro_precision": m["macro_precision"],
        "macro_recall": m["macro_recall"],
        "macro_f1": m["macro_f1"],
    })

overall_df = pd.DataFrame(overall_rows)
print(overall_df)


OVERALL SUMMARY BY LEVEL
  level  micro_precision  micro_recall  ...  macro_precision  macro_recall  macro_f1
0    h1         0.910424      0.896426  ...         0.893024      0.878508  0.885628
1    h2         0.849353      0.549843  ...         0.690966      0.367376  0.455470
2    h3         0.846742      0.604120  ...         0.776203      0.460549  0.559462
3    h4         0.875283      0.631807  ...         0.671517      0.409728  0.492235
4    h5         0.980715      0.980715  ...         0.938153      0.690123  0.762562

[5 rows x 7 columns]


In [74]:
use_nb = (meta_weights_df["coef_interaction"].abs() > 0)

meta_weights_df[use_nb]

,label,support,meta_type,coef_logit_prob,coef_nb_prob,coef_interaction,intercept,threshold
0,C1511,399,stacked,6.96886,-1.429824,2.855236,-6.904494,0.69


Section 3: Refit and Deployment
- Save the V6 model above to be ready to deploy on new data

In [99]:
# ------------------------------------------------------------
# V6 deployment helpers: fit inference bundle + predict on new data
# ------------------------------------------------------------

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB


def _safe_predict_proba_binary_for_inference(model, X):
    """Return P(y=1) robustly even for constant-label fits."""
    if hasattr(model, "classes_") and len(model.classes_) == 1:
        only_class = model.classes_[0]
        if only_class == 1:
            return np.ones(X.shape[0], dtype=float)
        return np.zeros(X.shape[0], dtype=float)

    probs = model.predict_proba(X)
    if probs.ndim == 1:
        return probs.astype(float)
    return probs[:, 1].astype(float)


def build_inference_bundle(
    Y_train_dict,
    X_train_df,
    X_train_vec,
    X_train_count,
    level_results,
    levels_order=None,
):
    """
    Fits and stores per-label base models needed for V6 inference on unseen data.

    Returns:
        dict with everything needed by predict_v6_on_new_data().
    """
    if levels_order is None:
        levels_order = list(level_results.keys())

    bundle = {
        "levels": levels_order,
        "models": {},
        "vectorizer": vectorizer,
        "count_vectorizer": count_vectorizer,
    }

    train_ids = X_train_df["id"]

    for level in levels_order:
        labels = list(level_results[level]["labels"])
        thresholds = level_results[level]["thresholds"]
        meta_models = level_results[level]["meta_models"]

        Y_train_level = Y_train_dict[level].loc[train_ids].copy()
        Y_train_level = Y_train_level.reindex(labels, axis=1)

        level_store = {
            "labels": labels,
            "thresholds": thresholds,
            "meta_models": meta_models,
            "base_models": {},
        }

        for label in labels:
            y = Y_train_level[label].to_numpy().astype(int)

            if np.unique(y).size < 2:
                # Constant label: logistic cannot fit one class.
                level_store["base_models"][label] = {
                    "logit": None,
                    "nb": None,
                    "constant_value": int(y[0]),
                }
                continue

            # Keep base training spec aligned with your V6 cell.
            logit_full = LogisticRegression(
                solver="lbfgs",
                max_iter=2000,
                class_weight="balanced",
            )
            logit_full.fit(X_train_vec, y)

            nb_full = MultinomialNB(alpha=1.0)
            nb_full.fit(X_train_count, y)

            level_store["base_models"][label] = {
                "logit": logit_full,
                "nb": nb_full,
            }

        bundle["models"][level] = level_store

    return bundle


def predict_on_new_data(new_df, bundle, return_probabilities=False):
    """
    Inputs:
        new_df: DataFrame with columns ['id', 'article']
        v6_bundle: output of build_v6_inference_bundle()

    Outputs:
        predictions_by_level: dict[level] -> DataFrame (index=id, columns=labels, values 0/1)
        combined_predictions: DataFrame (index=id, all labels columns, values 0/1)
        combined_probabilities (optional): DataFrame of meta probabilities
    """
    required_cols = {"id", "article"}
    if not required_cols.issubset(set(new_df.columns)):
        raise ValueError("new_df must contain 'id' and 'article' columns")

    df = new_df.copy()
    df["article"] = df["article"].fillna("").astype(str)

    X_new_vec = bundle["vectorizer"].transform(df["article"])
    X_new_count = bundle["count_vectorizer"].transform(df["article"])

    predictions_by_level = {}
    probs_by_level = {}

    for level in bundle["levels"]:
        level_store = bundle["models"][level]
        labels = level_store["labels"]
        thresholds = level_store["thresholds"]
        meta_models = level_store["meta_models"]
        base_models = level_store["base_models"]

        n_new = X_new_vec.shape[0]
        n_labels = len(labels)

        pred_matrix = np.zeros((n_new, n_labels), dtype=int)
        prob_matrix = np.zeros((n_new, n_labels), dtype=float)

        for j, label in enumerate(labels):
            meta_info = meta_models[label]

            if meta_info["type"] == "constant":
                probs = np.full(n_new, float(meta_info["value"]), dtype=float)
            else:
                logit_model = base_models[label]["logit"]
                nb_model = base_models[label]["nb"]

                logit_probs = _safe_predict_proba_binary_for_inference(logit_model, X_new_vec)
                nb_probs = _safe_predict_proba_binary_for_inference(nb_model, X_new_count)
                nb_probs_clip = np.sqrt(np.clip(nb_probs, 0.0, 1.0))

                if meta_info["type"] == "logistic_only":
                    probs = logit_probs
                else:
                    X_meta_new = np.column_stack([
                        logit_probs,
                        nb_probs_clip,
                        logit_probs * nb_probs_clip,
                    ])
                    probs = _safe_predict_proba_binary_for_inference(meta_info["model"], X_meta_new)

            threshold = float(thresholds[label])
            pred_matrix[:, j] = (probs >= threshold).astype(int)
            prob_matrix[:, j] = probs

        pred_df = pd.DataFrame(pred_matrix, index=df["id"].values, columns=labels)
        prob_df = pd.DataFrame(prob_matrix, index=df["id"].values, columns=labels)

        predictions_by_level[level] = pred_df
        probs_by_level[level] = prob_df

    combined_predictions = pd.concat(
        [predictions_by_level[level] for level in bundle["levels"]],
        axis=1,
    )
    combined_predictions = combined_predictions.loc[:, ~combined_predictions.columns.duplicated()]
    combined_predictions = combined_predictions.reindex(sorted(combined_predictions.columns), axis=1)

    if return_probabilities:
        combined_probabilities = pd.concat(
            [probs_by_level[level] for level in v6_bundle["levels"]],
            axis=1,
        )
        combined_probabilities = combined_probabilities.loc[:, ~combined_probabilities.columns.duplicated()]
        combined_probabilities = combined_probabilities.reindex(columns=combined_predictions.columns)
        return predictions_by_level, combined_predictions, combined_probabilities

    return predictions_by_level, combined_predictions


# Build once after training V6
inference_bundle = build_inference_bundle(
    Y_train_dict=Y_train,
    X_train_df=X_train,
    X_train_vec=X_train_vec,
    X_train_count=X_train_count,
    level_results=level_results,
    levels_order=levels,
)


# Example usage on a new dataframe with columns ['id', 'article']:
# new_data = pd.DataFrame({
#     "id": [900001, 900002],
#     "article": ["text of article one", "text of article two"],
# })
# pred_by_level, pred_all = predict_v6_on_new_data(new_data, v6_inference_bundle)
# print(pred_all.head())

In [101]:
pred_by_level, pred_all = predict_on_new_data(news_test, inference_bundle)
print(pred_all.head())

       C11         C12         C13         ...  M142        M143        MCAT      
26152           0           0           0  ...           0           0           0
26154           0           0           0  ...           0           0           0
26156           0           0           0  ...           0           0           0
26158           0           0           0  ...           0           0           0
26160           0           0           0  ...           0           0           0

[5 rows x 103 columns]
